In [ ]:
import os
RUN_MODE = 'FULL_REAL'
MOUNT_DRIVE = True            # outputs + resume files go to Drive, so they survive disconnects
DRIVE_DATA_ROOT = '/content/drive/MyDrive/AgriTrust_Data'   # contains CICIoT2023/ and UNSW_NB15/ (Option B)
USE_KAGGLE_MIRROR = True      # downloads CIC-IoT-2023 with your kaggle.json (recorded as a mirror in provenance)
SAVE_KAGGLE_KEY_TO_DRIVE = True   # keep a copy at MyDrive/kaggle.json so later runs need no upload (only if Drive is not shared)

assert RUN_MODE in ('SMOKE','QUICK_REAL','FULL_REAL')
os.environ['AGRITRUST_MODE'] = RUN_MODE
os.environ['AGRITRUST_SMOKE'] = '1' if RUN_MODE=='SMOKE' else '0'
os.environ['AGRITRUST_KAGGLE'] = '1' if USE_KAGGLE_MIRROR else '0'
print('RUN_MODE =', RUN_MODE, '| USE_KAGGLE_MIRROR =', USE_KAGGLE_MIRROR)

In [ ]:
import os, glob, shutil
if USE_KAGGLE_MIRROR and RUN_MODE!='SMOKE':
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    cands=[]
    for pattern in ['/root/.kaggle/kaggle.json','/content/*.json','/content/sample_data/*.json','/content/drive/MyDrive/kaggle*.json']:
        cands+=[p for p in glob.glob(pattern) if 'kaggle' in os.path.basename(p).lower()]
    if not cands:
        from google.colab import files
        print('kaggle.json not found. Click "Choose Files" below and pick kaggle.json from your computer.')
        up=files.upload()
        cands=[os.path.join('/content',n) for n in up if n.lower().endswith('.json')]
    if not cands:
        raise RuntimeError('No kaggle.json provided. Download it from kaggle.com -> Settings -> API -> Create New Token, then rerun this cell.')
    os.makedirs('/root/.kaggle',exist_ok=True)
    if os.path.abspath(cands[0])!='/root/.kaggle/kaggle.json': shutil.copy(cands[0],'/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json',0o600)
    if not os.path.exists('/content/kaggle.json'): shutil.copy('/root/.kaggle/kaggle.json','/content/kaggle.json')
    if SAVE_KAGGLE_KEY_TO_DRIVE and not os.path.exists('/content/drive/MyDrive/kaggle.json'):
        shutil.copy('/root/.kaggle/kaggle.json','/content/drive/MyDrive/kaggle.json'); print('Saved a copy to MyDrive/kaggle.json')
    print('Kaggle key installed from',cands[0])
else:
    print('Kaggle key not needed (USE_KAGGLE_MIRROR is False or RUN_MODE is SMOKE).')

In [ ]:
import sys, shutil, platform
from pathlib import Path
import torch
print('Python  :', sys.version.split()[0], '|', platform.platform())
print('PyTorch :', torch.__version__)
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print(f'GPU     : {p.name} ({p.total_memory/1024**3:.1f} GB)')
else:
    print('GPU     : none (CPU runtime)')
try:
    ram=os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1024**3; print(f'RAM     : {ram:.1f} GB')
except Exception: pass
du=shutil.disk_usage('/content' if Path('/content').exists() else '/'); print(f'Disk    : {du.free/1024**3:.1f} GB free')
if RUN_MODE=='FULL_REAL' and not torch.cuda.is_available():
    print('WARNING: FULL_REAL on CPU will be very slow. Runtime -> Change runtime type -> T4 GPU.')

In [ ]:
import glob, zipfile, shutil
IN_COLAB = Path('/content').exists()
if IN_COLAB and MOUNT_DRIVE and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive; drive.mount('/content/drive')

def _csvs(root): return sorted(glob.glob(str(Path(root)/'**'/'*.csv'),recursive=True)) if Path(root).exists() else []
def _unzip_all(src_root,dst):
    for z in sorted(Path(src_root).rglob('*.zip')):
        print('unzipping',z,'->',dst); zipfile.ZipFile(z).extractall(dst)

if RUN_MODE!='SMOKE':
    base=Path('/content/data') if IN_COLAB else Path('/mnt/data/data')
    CIC_LOCAL=base/'CICIOT2023'; UNSW_LOCAL=base/'UNSW_NB15'
    for p in (CIC_LOCAL,UNSW_LOCAL): p.mkdir(parents=True,exist_ok=True)
    drive_cic=Path(DRIVE_DATA_ROOT)/'CICIoT2023'; drive_unsw=Path(DRIVE_DATA_ROOT)/'UNSW_NB15'


    if not _csvs(CIC_LOCAL): _unzip_all(CIC_LOCAL,CIC_LOCAL)
    if _csvs(CIC_LOCAL):
        cic_glob=str(CIC_LOCAL/'**'/'*.csv'); cic_src=f'local {CIC_LOCAL}'
    elif _csvs(drive_cic):
        cic_glob=str(drive_cic/'**'/'*.csv'); cic_src=f'Google Drive {drive_cic}'
    elif drive_cic.exists() and list(drive_cic.rglob('*.zip')):
        _unzip_all(drive_cic,CIC_LOCAL); cic_glob=str(CIC_LOCAL/'**'/'*.csv'); cic_src=f'zip from {drive_cic}'
    else:
        cic_glob=str(CIC_LOCAL/'**'/'*.csv'); cic_src='NOT FOUND'
    n_cic=len(glob.glob(cic_glob,recursive=True))
    os.environ['AGRITRUST_CIC_GLOB']=cic_glob; os.environ['AGRITRUST_CIC_SOURCE']=cic_src

       def _find(name):
        for root in (UNSW_LOCAL,drive_unsw):
            hits=[p for p in Path(root).rglob('*.csv') if p.name.lower()==name.lower()] if Path(root).exists() else []
            if hits: return hits[0]
        return None
    if _find('UNSW_NB15_training-set.csv') is None:
        for root in (UNSW_LOCAL,drive_unsw):
            if Path(root).exists(): _unzip_all(root,UNSW_LOCAL)
    utr_p=_find('UNSW_NB15_training-set.csv'); ute_p=_find('UNSW_NB15_testing-set.csv')
    unsw_src='local/Drive' if (utr_p and ute_p) else 'NOT FOUND'
        UNSW_KAGGLE_SLUGS=['alextamboli/unsw-nb15','mrwellsdavid/unsw-nb15']
    if 'kaggle_download' not in globals():
        def kaggle_download(slug,dst):
            import subprocess, sys
            dst=Path(dst); dst.mkdir(parents=True,exist_ok=True)
            try:
                if shutil.which('kaggle') is None:
                    subprocess.run([sys.executable,'-m','pip','install','-q','kaggle'],check=False)
                r=subprocess.run(['kaggle','datasets','download','-d',slug,'-p',str(dst),'--unzip'],capture_output=True,text=True,timeout=3600)
                print(f'kaggle {slug}: exit code {r.returncode}',(r.stderr or '')[-300:])
                return r.returncode==0
            except Exception as e:
                print('kaggle download failed for',slug,':',e); return False
    if (utr_p is None or ute_p is None) and USE_KAGGLE_MIRROR and RUN_MODE=='FULL_REAL':
        for slug in UNSW_KAGGLE_SLUGS:
            print('Attempting UNSW-NB15 Kaggle download:',slug)
            kaggle_download(slug,UNSW_LOCAL/'_kaggle'/slug.replace('/','__'))
            utr_p=_find('UNSW_NB15_training-set.csv'); ute_p=_find('UNSW_NB15_testing-set.csv')
            if utr_p and ute_p:
                unsw_src=f'Kaggle convenience mirror {slug} (verify against the official UNSW release)'; break
            print('  mirror',slug,'did not contain both partition files; trying the next one')
    os.environ['AGRITRUST_UNSW_SOURCE']=unsw_src
    if utr_p: os.environ['AGRITRUST_UNSW_TRAIN']=str(utr_p)
    if ute_p: os.environ['AGRITRUST_UNSW_TEST']=str(ute_p)

    print(f'CIC-IoT-2023 : {n_cic} CSV file(s) | source: {cic_src}')
    print(f'UNSW-NB15    : train={utr_p} | test={ute_p} | source: {unsw_src}')
    if n_cic==0 and not USE_KAGGLE_MIRROR:
        raise RuntimeError('No CIC-IoT-2023 CSVs found. Put the official CSVs (or zip) in '
                           f'{CIC_LOCAL} or {drive_cic}, or set USE_KAGGLE_MIRROR=True in Step 0a.')
    if RUN_MODE=='FULL_REAL' and not (utr_p and ute_p):
        print('WARNING: UNSW-NB15 files not found - the external replication (Table 21) will be marked NOT_RUN.')
else:
    print('SMOKE mode: dataset staging skipped (synthetic data).')

In [ ]:
import os, re, json, math, time, random, copy, warnings, hashlib, glob, shutil, subprocess, sys
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    log_loss, roc_curve, precision_recall_curve, auc, precision_score, recall_score
)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')

SMOKE = os.getenv('AGRITRUST_SMOKE','0') == '1'
RUN_MODE = os.getenv('AGRITRUST_MODE','SMOKE' if SMOKE else 'QUICK_REAL')   # set in Step 0a
assert RUN_MODE in ('SMOKE','QUICK_REAL','FULL_REAL'), RUN_MODE

CONFIG = {
    'SEED': 42,
    'QUICK_RUN': RUN_MODE!='FULL_REAL',

    'DATA_MODE': 'SYNTHETIC_SMOKE' if SMOKE else 'REAL',   # REAL never falls back to synthetic data
    'PRIMARY_DATASET': 'CICIOT2023',
    'CICIOT2023_GLOB': os.getenv('AGRITRUST_CIC_GLOB','/content/data/CICIOT2023/**/*.csv'),
    'CICIOT2023_FALLBACK_GLOB': None,
    'CICIOT2023_SOURCE_NOTE': os.getenv('AGRITRUST_CIC_SOURCE','local /content/data/CICIOT2023'),
    'USE_KAGGLE_MIRROR': os.getenv('AGRITRUST_KAGGLE','0')=='1',
    'CICIOT2023_OFFICIAL_PAGE': 'https://www.unb.ca/cic/datasets/iotdataset-2023.html',
    'CICIOT2023_KAGGLE_SLUG': 'madhavmalhotra/unb-cic-iot-dataset',

    'RUN_EXTERNAL_UNSW': False,
    'UNSW_TRAIN': os.getenv('AGRITRUST_UNSW_TRAIN','/content/data/UNSW_NB15/UNSW_NB15_training-set.csv'),
    'UNSW_TEST': os.getenv('AGRITRUST_UNSW_TEST','/content/data/UNSW_NB15/UNSW_NB15_testing-set.csv'),
    'UNSW_FED_ROUNDS': 10,
    'UNSW_SOURCE_NOTE': os.getenv('AGRITRUST_UNSW_SOURCE','local'),
    'UNSW_POISON_SCENARIOS': [('sign_flip',0.30),('model_replace',0.30)],
    'DROP_EXACT_DUPLICATES': True,
    'UNSW_OFFICIAL_PAGE': 'https://research.unsw.edu.au/projects/unsw-nb15-dataset',

    'MAX_ROWS': 60000 if not SMOKE else 1600,
    'MAX_FEATURES': 46 if not SMOKE else 10,

    'VAL_FRAC_FROM_TRAIN': 0.10,
    'CAL_FRAC': 0.15,
    'TEST_FRAC': 0.20,

    'N_CLIENTS': 10 if not SMOKE else 4,
    'DIRICHLET_ALPHA': 0.25,
    'ROUNDS': 30 if not SMOKE else 2,
    'LOCAL_EPOCHS': 2 if not SMOKE else 1,
    'BATCH_SIZE': 256 if not SMOKE else 64,
    'LR': 1e-3,
    'WEIGHT_DECAY': 1e-4,
    'FEDPROX_MU': 1e-2,
    'D_MODEL': 128 if not SMOKE else 16,
    'DROPOUT': 0.10,
    'CLIENT_FRACTION': 0.80,

    'MAX_STALENESS': 6,
    'STALENESS_LAMBDA': 0.45,
    'TRUST_RHO': 0.75,
    'TRUST_FLOOR': 0.05,
    'TRUST_ALPHA': 0.05,
    'DEFAULT_TRUST_THRESHOLD': 0.24,
    'TRUST_WARMUP_ROUNDS': 10 if not SMOKE else 3,
    'CLIP_MULTIPLIER': 2.5,
    'TRIM_FRAC': 0.20,
    'MULTIKRUM_F': 2,

    'POISON_RATIOS': [0.25] if SMOKE else [0.10,0.20,0.30,0.40],
    'POISON_TYPES': ['sign_flip'] if SMOKE else [
        'sign_flip','model_replace','gaussian','collusive_scale','adaptive_slow_drift','alie_jitter'
    ],
    'ADAPTIVE_DRIFT_SCALE': 0.20,
    'COLLUSIVE_SCALE': 6.0,

    'NOISE_LEVELS': [0.0,0.25] if SMOKE else [0.0,0.10,0.25,0.50,0.75],
    'ROBUSTNESS_DROPOUT': [0.0,0.40] if SMOKE else [0.0,0.20,0.40,0.60],
    'ROBUSTNESS_STALENESS': [0,2] if SMOKE else [0,1,2,4,6],

    'TAU_SENSITIVITY': [0.5,0.85] if SMOKE else [0.50,0.60,0.70,0.80,0.85,0.90,0.95],   # tau_cal is appended automatically
    'LAMBDA_SENSITIVITY': [0.25,0.75] if SMOKE else [0.00,0.10,0.25,0.45,0.75,1.00,1.50],   # 0.0 = no freshness decay; 1.5 guards the upper edge

    'RUN_MULTI_SEED': False,
    'PAPER_SEEDS': [7,19,41,73,101,131,157,181,211,239],

        'OUTDIR': (f'/content/drive/MyDrive/AgriTrust_Results_{RUN_MODE}' if Path('/content/drive/MyDrive').exists()
              else (f'/content/agritrust_acfl_outputs_{RUN_MODE}' if Path('/content').exists()
                    else f'/mnt/data/agritrust_acfl_outputs_{RUN_MODE}'))
}

CONFIG.update({
    'TRUST_THRESHOLD_CAP': 0.95,
    'GATE_MODE': 'hybrid',          # 'hybrid' (drift-normalised) | 'absolute' | 'relative' | 'off'
    'GATE_KAPPA': 3.0,              # only used by the optional 'relative' gate
    'GATE_MAD_FLOOR': 0.02,         # prevents a near-zero MAD from quarantining tiny deviations
    'MIN_QUORUM_FRAC': 0.50,        # never admit fewer than this fraction of the round's clients
    'WARM_START_ROUNDS': 10 if not SMOKE else 1,
    'POISON_ROUNDS': 6 if not SMOKE else 1,
    'AUDIT_ROUNDS': 5 if not SMOKE else 1,
    'ABLATION_ROUNDS': 6 if not SMOKE else 1,
    'ABLATION_SCENARIOS': [('none',0.0),('sign_flip',0.25)] if SMOKE else
                          [('none',0.0),('sign_flip',0.30),('model_replace',0.30),('collusive_scale',0.30),('adaptive_slow_drift',0.30)],
    'SENS_ROUNDS': 5 if not SMOKE else 1,
    'SENS_SCENARIOS': [('none',0.0),('sign_flip',0.25)] if SMOKE else [('none',0.0),('sign_flip',0.30)],
    'POISON_SEEDS_N': 10,
    'TRUST_UPDATE': 'asymmetric',
    'SELECT_TRUST_UPDATE_ON_VALIDATION': True,
    'TRUST_RHO_DOWN': 0.30,          # memory weight when instant trust falls clearly below history (fast drop)
    'TRUST_RHO_UP': 0.85,            # memory weight otherwise (slow recovery)
    'TRUST_DEADBAND': 0.05,          # dips smaller than this use the slow weight -> no downward ratchet on noisy benign clients
    'TRUST_CAL_BURNIN': 2,           # first warm-up rounds excluded from calibration (trust still falling from its initial 1.0)
    'MAX_BENIGN_FQ_FOR_SELECTION': 0.10,
    'EXCLUDE_OTHER_CLASS': True,     # True removes the rare 'Other' grab-bag class from the task (reported in Table 00/01b)
    # bounded exclusion (probation): after PROBATION_AFTER consecutive quarantined participations a client is
    # admitted once at PROBATION_WEIGHT x its normal (clipped, trust-weighted) weight. Caps benign lock-out by construction.
    'PROBATION_AFTER': 3,
    'PROBATION_WEIGHT': 0.5,
    # similarity (sybil) signal, after FoolsGold (Fung, Yoon & Beschastnikh, RAID 2020): independently trained
    # honest gateways never submit near-identical updates; colluding copies do. Near-duplicates (a) are penalised and
    # (b) count as ONE vote when the consensus direction and norm are computed.
    'SIM_START': 0.99,               # max pairwise cosine at which the penalty starts
    'SIM_FULL': 0.999,               # max pairwise cosine at which the penalty is complete
    'SIM_FLOOR': 0.05,               # similarity score of an exact duplicate
    'ALIE_JITTER': 0.30,             # adaptive attacker: per-copy noise (x ALIE norm) to evade the similarity signal
    'SELECT_SEEDS_N': 3 if not SMOKE else 1,   # seeds averaged in rule / lambda selection
    # client-size cap: capacity-constrained Dirichlet split, no client above MAX_CLIENT_SHARE of the rows.
    # uncapped split gave one DDoS-only client 66% of all training rows, so data-size-weighted averaging
    # (FedAvg, Async-Stale, RawTrust) effectively trained on that one farm and stalled at ~0.36 Macro-F1.
    'MAX_CLIENT_SHARE': 0.25,
    'RUN_DOMINANT_CLIENT_SCENARIO': True,      # the uncapped split as a separate, clean comparison (Table 13b)
    'ABLATION_SEEDS_N': 1,                     # 3 in FULL_REAL (set below)
    #  lambda is selected on VALIDATION before any main run (never on test)
    # lambda FIXED by design at STALENESS_LAMBDA (0.45: an update 2 rounds late keeps ~40% weight). Validation could
    # not distinguish lambda values (all within noise; the edge value won twice), and a steep decay would sideline
    # intermittent rural gateways. The full lambda curve is still reported in the sensitivity table.
    'SELECT_LAMBDA_ON_VALIDATION': False,
    'LAMBDA_SELECT_ROUNDS': 10 if not SMOKE else 1,
    'LAMBDA_SELECT_SCENARIOS': [('none',0.0),('sign_flip',0.25)] if SMOKE else [('none',0.0),('sign_flip',0.30)],
    # genuine stealthy slow-drift adversary: ramped ALIE (Baruch et al., NeurIPS 2019); z grows each round
    'DRIFT_Z0': 0.30, 'DRIFT_Z_STEP': 0.15, 'DRIFT_Z_MAX': 1.50,
    'MULTISEED_POISON_SCENARIOS': [('sign_flip',0.25)] if SMOKE else
                                  [('sign_flip',0.30),('model_replace',0.30),('collusive_scale',0.30),('adaptive_slow_drift',0.30)],
})

if not CONFIG['QUICK_RUN'] and not SMOKE:
    CONFIG.update({
        'MAX_ROWS': 180000,
        'MAX_FEATURES': 60,
        'N_CLIENTS': 10,
        'ROUNDS': 40,
        'LOCAL_EPOCHS': 2,
        'BATCH_SIZE': 512,
        'D_MODEL': 128,
        'RUN_MULTI_SEED': True,
        'RUN_EXTERNAL_UNSW': True,
        'UNSW_FED_ROUNDS': 30,
        'LAMBDA_SELECT_ROUNDS': 15,
        'ABLATION_SEEDS_N': 3,
        'WARM_START_ROUNDS': 12,
        'POISON_ROUNDS': 10,
        'AUDIT_ROUNDS': 6,
        'ABLATION_ROUNDS': 8,
        'SENS_ROUNDS': 6
    })

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_all(seed):
    seed=int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(CONFIG['SEED'])
print('Device:', DEVICE)
print('RUN_MODE:', RUN_MODE, '| QUICK_RUN:', CONFIG['QUICK_RUN'], '| DATA_MODE:', CONFIG['DATA_MODE'], '| RUN_MULTI_SEED:', CONFIG['RUN_MULTI_SEED'])
print('Outputs ->', CONFIG['OUTDIR'])
print('Primary dataset:', CONFIG['PRIMARY_DATASET'])
if DEVICE.type == 'cpu' and not SMOKE:
    print('WARNING: GPU strongly recommended for the full publication run.')


In [ ]:
# Cell 2 — Output tree, save helpers and run manifest
OUT=Path(CONFIG['OUTDIR']); TAB=OUT/'tables'; FIG=OUT/'figures'; MOD=OUT/'models'; LOG=OUT/'logs'
for p in [OUT,TAB,FIG,MOD,LOG]: p.mkdir(parents=True,exist_ok=True)
RUN_START=time.time()

def save_table(df,name):
    p=TAB/f'{name}.csv'; df.to_csv(p,index=False); print('saved',p); return df

def savefig(name):
    plt.tight_layout()
    for ext in ('png','pdf'):
        plt.savefig(FIG/f'{name}.{ext}',dpi=300,bbox_inches='tight')
    plt.close(); print('saved figure',name)

def cfg_hash():
    return hashlib.sha256(json.dumps(CONFIG,sort_keys=True,default=str).encode()).hexdigest()[:12]
print('Run ID:',cfg_hash())


In [ ]:

# Cell 3 — Dataset acquisition and provenance
def _read_csv_limited(path, max_rows=None):
    try:
        return pd.read_csv(path,nrows=max_rows,low_memory=False)
    except Exception:
        return pd.read_csv(path,nrows=max_rows,low_memory=False,encoding_errors='ignore')

FILE_MANIFEST=[]
def _head_sha256(path,nbytes=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as fh: h.update(fh.read(nbytes))
    return h.hexdigest()[:16]

def load_local_csvs(pattern,max_rows):
    """EVERY file contributes. N = ceil(MAX_ROWS / #files) head rows per file (v2 used max(1000, MAX_ROWS/#files),
    which exhausted the row budget after the first 60 of 169 CIC-IoT-2023 files). Each file gets a manifest row -
    including failed / empty files - with its size, rows read, column count and a SHA-256 of its first MiB."""
    FILE_MANIFEST.clear()
    files=sorted(glob.glob(pattern,recursive=True)) if pattern else []
    if not files:
        return None,[]
    per_file=max(1,-(-int(max_rows)//len(files)))          # ceil division; at least 1 row per file
    root=os.path.commonpath(files) if len(files)>1 else os.path.dirname(files[0])
    parts=[]; remaining=max_rows
    for f in files:
        rec={'File':os.path.relpath(f,root),'Bytes':os.path.getsize(f),'RowsRead':0,'Columns':np.nan,
             'Head1MiB_SHA256':_head_sha256(f),'SamplingRule':f'first {per_file} rows of every file = ceil(MAX_ROWS/#files)','Status':'ok'}
        if remaining<=0:
            rec['Status']='not read (row cap reached)'; FILE_MANIFEST.append(rec); continue
        try:
            d=_read_csv_limited(f,min(remaining,per_file)); rec['RowsRead']=len(d); rec['Columns']=d.shape[1]
            if len(d): parts.append(d); remaining-=len(d)
            else: rec['Status']='empty'
        except Exception as e:
            rec['Status']='failed: '+str(e)[:80]; print('skip',f,str(e)[:120])
        FILE_MANIFEST.append(rec)
    return (pd.concat(parts,ignore_index=True) if parts else None),files

def try_kaggle_ciciot2023():
    slug=CONFIG['CICIOT2023_KAGGLE_SLUG']
    dst=Path('/content/data/CICIOT2023')
    dst.mkdir(parents=True,exist_ok=True)
    if list(dst.rglob('*.csv')):
        return
    if not Path('/root/.kaggle/kaggle.json').exists() and not Path('/content/kaggle.json').exists() and Path('/content/drive/MyDrive/kaggle.json').exists():
        shutil.copy('/content/drive/MyDrive/kaggle.json','/content/kaggle.json')
    if not Path('/root/.kaggle/kaggle.json').exists() and not Path('/content/kaggle.json').exists():
        print('Kaggle credentials not found; skipping convenience download.')
        return
    if Path('/content/kaggle.json').exists():
        Path('/root/.kaggle').mkdir(parents=True,exist_ok=True)
        shutil.copy('/content/kaggle.json','/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json',0o600)
    subprocess.run([sys.executable,'-m','pip','-q','install','kaggle'],check=False)
    print('Attempting CIC-IoT-2023 convenience download:',slug)
    subprocess.run(['kaggle','datasets','download','-d',slug,'-p',str(dst),'--unzip'],check=False)

def make_synthetic_smoke(n=1600,p=10,k=6,seed=42):
    rng=np.random.default_rng(seed)
    X=rng.normal(size=(n,p)); W=rng.normal(size=(p,k))
    logits=X@W + .35*rng.normal(size=(n,k))
    yy=logits.argmax(1); names=['Benign','DDoS','DoS','Mirai','Recon','Spoofing'][:k]
    d=pd.DataFrame(X,columns=[f'f{i:02d}' for i in range(p)])
    d['label']=[names[i] for i in yy]
    return d

raw=None; primary_files=[]; provenance=''
if CONFIG['DATA_MODE'] in ('AUTO','REAL'):
    raw,primary_files=load_local_csvs(CONFIG['CICIOT2023_GLOB'],CONFIG['MAX_ROWS'])
    if raw is None and CONFIG.get('CICIOT2023_FALLBACK_GLOB'):
        raw,primary_files=load_local_csvs(CONFIG['CICIOT2023_FALLBACK_GLOB'],CONFIG['MAX_ROWS'])
    KAGGLE_USED=False
    if raw is None and Path('/content').exists() and CONFIG.get('USE_KAGGLE_MIRROR',False):
        try_kaggle_ciciot2023(); KAGGLE_USED=True
        raw,primary_files=load_local_csvs(str(Path('/content/data/CICIOT2023')/'**'/'*.csv'),CONFIG['MAX_ROWS'])
    if raw is not None:
        src_note=(f'Kaggle convenience mirror {CONFIG["CICIOT2023_KAGGLE_SLUG"]} (verify against official files)'
                  if KAGGLE_USED else CONFIG.get('CICIOT2023_SOURCE_NOTE','local files'))
        provenance=(f'REAL: CIC-IoT-2023; {sum(r["RowsRead"]>0 for r in FILE_MANIFEST)} of {len(primary_files)} CSV files contributed rows; source: {src_note}. '
                    f'Official source: {CONFIG["CICIOT2023_OFFICIAL_PAGE"]}')

if raw is None:
    if CONFIG['DATA_MODE']=='REAL':
        raise RuntimeError(
            'REAL mode found no CIC-IoT-2023 CSVs at '+str(CONFIG['CICIOT2023_GLOB'])+'. '
            'Run Step 0c, place the official CSVs (or their zip) in /content/data/CICIOT2023/ or '
            '<DRIVE_DATA_ROOT>/CICIoT2023/, or set USE_KAGGLE_MIRROR=True in Step 0a.'
        )
    raw=make_synthetic_smoke(CONFIG['MAX_ROWS'],CONFIG['MAX_FEATURES'],6,CONFIG['SEED'])
    provenance='SYNTHETIC_SMOKE: pipeline verification only; NOT manuscript evidence.'

print('PRIMARY RAW:',raw.shape)
print(provenance)

dataset_manifest=pd.DataFrame([
    {'Role':'Primary','Dataset':'CIC-IoT-2023','OfficialSource':CONFIG['CICIOT2023_OFFICIAL_PAGE'],
     'LocalFiles':len(primary_files),'RowsLoaded':len(raw),
     'Use':'Main AgriTrust training/validation/testing and federated stress experiments'},
    {'Role':'External','Dataset':'UNSW-NB15','OfficialSource':CONFIG['UNSW_OFFICIAL_PAGE'],
     'LocalFiles':int(Path(CONFIG['UNSW_TRAIN']).exists())+int(Path(CONFIG['UNSW_TEST']).exists()),
     'RowsLoaded':0,'Use':'Separate external validation; schema is re-fitted and never pooled with CIC-IoT-2023'}
])
save_table(dataset_manifest,'Table_00a_Dataset_Manifest')
save_table(pd.DataFrame(FILE_MANIFEST) if FILE_MANIFEST else pd.DataFrame([{'File':'(none: synthetic smoke)'}]),'Table_00b_Primary_File_Manifest')


In [ ]:
def find_label_col(df):
    for c in ['label','Label','attack','Attack','attack_type','Attack_type','Attack Type','class','Class','target','Target']:
        if c in df.columns: return c
    obj=[c for c in df.columns if str(df[c].dtype)=='object']
    obj=sorted(obj,key=lambda c:df[c].nunique(dropna=True))
    for c in obj:
        if 2<=df[c].nunique(dropna=True)<=250: return c
    raise RuntimeError('Could not infer label column. Rename the target column to label.')

def coarse_label(v):
    s=str(v).strip().lower()
    if 'benign' in s or s in {'normal','0'}: return 'Benign'
    if 'ddos' in s: return 'DDoS'
    if re.search(r'(^|[^d])dos([^a-z]|$)',s): return 'DoS'
    if 'mirai' in s: return 'Mirai'
    if 'spoof' in s or 'arp' in s: return 'Spoofing'
    if 'recon' in s or 'scan' in s or 'portscan' in s: return 'Recon'
    if 'brute' in s or 'dictionary' in s or 'password' in s: return 'BruteForce'
    if 'web' in s or 'xss' in s or 'sql' in s or 'upload' in s: return 'Web'
    return str(v).strip()[:48] or 'Other'

label_col=find_label_col(raw); df=raw.copy()
# Exact duplicate rows are removed BEFORE splitting so the same record cannot sit in both train and test.
n_rows_loaded=len(df)
if CONFIG.get('DROP_EXACT_DUPLICATES',True): df=df.drop_duplicates().reset_index(drop=True)
n_exact_dupes=n_rows_loaded-len(df)
df['_target']=df[label_col].map(coarse_label)
vc=df['_target'].value_counts(); small=set(vc[vc<max(15,int(.001*len(df)))].index); df.loc[df['_target'].isin(small),'_target']='Other'
# Label-harmonization table is built BEFORE any exclusion so excluded raw labels stay documented.
label_map=(df.groupby([label_col,'_target']).size().reset_index(name='Rows')
             .rename(columns={label_col:'RawLabel','_target':'HarmonizedClass'}).sort_values(['HarmonizedClass','RawLabel']))
label_map['UsedInTask']=~(bool(CONFIG.get('EXCLUDE_OTHER_CLASS',False)) & (label_map.HarmonizedClass=='Other'))
n_other_excluded=0
if CONFIG.get('EXCLUDE_OTHER_CLASS',False):
    n_other_excluded=int((df['_target']=='Other').sum()); df=df[df['_target']!='Other'].reset_index(drop=True)
    print('Excluded rare Other class rows:',n_other_excluded)
exclude={label_col,'_target'}; num=[]
for c in df.columns:
    if c in exclude: continue
    if pd.api.types.is_numeric_dtype(df[c]) and not any(t in c.lower() for t in ['label','target','class_id']): num.append(c)
if len(num)<4: raise RuntimeError(f'Need >=4 numeric features; found {len(num)}')
classes=sorted(df['_target'].astype(str).unique()); class_to_id={c:i for i,c in enumerate(classes)}
y=np.array([class_to_id[v] for v in df['_target'].astype(str)],dtype=np.int64); idx=np.arange(len(df))
# Split FIRST; every data-dependent preprocessing decision below is fitted on training rows only.
train_pool_idx,tmp_idx=train_test_split(
    idx,test_size=CONFIG['CAL_FRAC']+CONFIG['TEST_FRAC'],
    stratify=y,random_state=CONFIG['SEED']
)
relative_test=CONFIG['TEST_FRAC']/(CONFIG['CAL_FRAC']+CONFIG['TEST_FRAC'])
cal_idx,test_idx=train_test_split(
    tmp_idx,test_size=relative_test,stratify=y[tmp_idx],random_state=CONFIG['SEED']+1
)
train_idx,val_idx=train_test_split(
    train_pool_idx,test_size=CONFIG['VAL_FRAC_FROM_TRAIN'],
    stratify=y[train_pool_idx],random_state=CONFIG['SEED']+17
)
X0=df[num].replace([np.inf,-np.inf],np.nan); Xtr0=X0.iloc[train_idx]
miss=Xtr0.isna().mean(); num=[c for c in num if miss[c]<=.40 and Xtr0[c].nunique(dropna=True)>1]
vari=Xtr0[num].var(numeric_only=True).replace([np.inf,-np.inf],np.nan).fillna(0).sort_values(ascending=False)
num=list(vari.head(CONFIG['MAX_FEATURES']).index); X0=X0[num]
train_medians=X0.iloc[train_idx].median()
for c in num: X0[c]=X0[c].fillna(train_medians[c])
# Cross-split audit: test rows whose (selected) feature vector is identical to some training row.
_hv=pd.util.hash_pandas_object(X0,index=False).values
n_test_feature_overlap=int(np.isin(_hv[test_idx],_hv[train_idx]).sum())
save_table(label_map,'Table_01b_Label_Harmonization')
scaler=StandardScaler().fit(X0.iloc[train_idx]); X=scaler.transform(X0).astype('float32')
profile=pd.DataFrame({'Class':classes,'Count':[int((y==i).sum()) for i in range(len(classes))]}); profile['Percent']=100*profile.Count/len(y)
save_table(profile,'Table_01_Class_Profile')
save_table(pd.DataFrame({'Feature':num,'TrainMean':X0.iloc[train_idx].mean().values,'TrainStd':X0.iloc[train_idx].std().values,'MissingFraction':[miss[c] for c in num]}),'Table_02_Feature_Profile')
summary=pd.DataFrame([{'RowsLoaded':n_rows_loaded,'OtherClassRowsExcluded':n_other_excluded,'ExactDuplicatesRemoved':n_exact_dupes,'TestRowsWithIdenticalTrainFeatures':n_test_feature_overlap,'Rows':len(df),'Features':len(num),'Classes':len(classes),'Train':len(train_idx),'Validation':len(val_idx),'Calibration':len(cal_idx),'Test':len(test_idx),'Provenance':provenance,'RunID':cfg_hash()}])
save_table(summary,'Table_00_Run_and_Data_Summary'); print(summary.to_string(index=False))


In [ ]:
checks_d=[]
def dcheck(name,ok,detail,severity='ERROR'):
    checks_d.append({'Check':name,'Passed':bool(ok),'Severity':'OK' if ok else severity,'Detail':detail})
real_data_loaded=provenance.startswith('REAL')
dcheck('Real CIC-IoT-2023 data loaded (not synthetic)',real_data_loaded or SMOKE,provenance[:160])
if real_data_loaded:
    _man=pd.DataFrame(FILE_MANIFEST); n_contrib=int((_man.RowsRead>0).sum()) if len(_man) else 0
    dcheck('Number of primary CSV files found',len(primary_files)>=10,f'{len(primary_files)} file(s) found','WARN')
    dcheck('Every found file contributed rows (>= 95%)',n_contrib>=0.95*len(primary_files),
           f'{n_contrib} of {len(primary_files)} files contributed rows; statuses: {_man.Status.str.split(":").str[0].value_counts().to_dict() if len(_man) else {}}')
    _ncols=_man.loc[_man.RowsRead>0,'Columns'].nunique() if len(_man) else 0
    dcheck('All contributing files share one column schema',_ncols<=1,f'{_ncols} distinct column counts across files','WARN')
    dcheck('Label column found',label_col is not None,f'label column = {label_col!r}')
    dcheck('Benign class present after harmonization','Benign' in classes,f'classes = {classes}')
    dcheck('At least 5 harmonized classes',len(classes)>=5,f'{len(classes)} classes')
    other=float((y==class_to_id['Other']).mean()) if 'Other' in class_to_id else 0.0
    dcheck("'Other' (unmapped/rare) class share <= 5%",other<=.05,f'{100*other:.2f}% of rows','WARN')
    dcheck('At least 20 numeric features retained',len(num)>=20,f'{len(num)} features','WARN')
    dcheck('Smallest class has >= 50 training rows',np.bincount(y[train_idx],minlength=len(classes)).min()>=50,
           f'min train rows per class = {int(np.bincount(y[train_idx],minlength=len(classes)).min())}','WARN')
    frac=n_test_feature_overlap/max(1,len(test_idx))
    dcheck('Test rows identical to a training row (after dedup) <= 1%',frac<=.01,
           f'{n_test_feature_overlap} of {len(test_idx)} test rows ({100*frac:.2f}%); exact duplicates removed = {n_exact_dupes}','WARN')
    dcheck('Primary source is official/local (not a mirror)','Kaggle' not in provenance,
           provenance.split('source: ')[1].split('. Official')[0] if 'source: ' in provenance else provenance[:120],'WARN')
data_checks=pd.DataFrame(checks_d); save_table(data_checks,'Table_00c_Data_Integrity_Checks'); print(data_checks.to_string(index=False))
errs=data_checks[(data_checks.Severity=='ERROR')]
if len(errs) and CONFIG['DATA_MODE']=='REAL':
    raise RuntimeError('Data integrity gate failed:\n'+errs.to_string(index=False))

In [ ]:
import pickle
RESUME_KEY=hashlib.sha256((cfg_hash()+'|'+provenance+'|'+str(len(train_idx))+'|'+str(len(test_idx))+'|'
                          +str(int(np.asarray(y)[train_idx].sum()))+'|'+','.join(num)).encode()).hexdigest()[:12]
CKPT=OUT/'checkpoints'/RESUME_KEY; CKPT.mkdir(parents=True,exist_ok=True)
print('Resume key:',RESUME_KEY,'| checkpoints:',CKPT,'| finished stages found:',sorted(p.stem for p in CKPT.glob('*.pkl')))

def run_stage(name,fn):
    """Run fn() once and checkpoint its result; on a later run, load it instead. Run-registry entries written by the
    stage are stored with it and restored on load, so the configuration diagnostics stay complete."""
    f=CKPT/f'{name}.pkl'
    if f.exists():
        try:
            with open(f,'rb') as fh: obj=pickle.load(fh)
            globals()['RUN_REGISTRY'].extend(obj.get('_registry',[]))
            print(f'RESUME: stage "{name}" loaded from checkpoint (computed earlier in {obj.get("seconds",0)/60:.1f} min)')
            return obj['result']
        except Exception as e:
            print(f'checkpoint for "{name}" unreadable ({e}); recomputing')
    n0=len(globals()['RUN_REGISTRY']); t0=time.time()
    res=fn()
    tmp=f.with_suffix('.tmp')
    with open(tmp,'wb') as fh:
        pickle.dump({'result':res,'_registry':globals()['RUN_REGISTRY'][n0:],'seconds':time.time()-t0},fh,protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp,f)
    print(f'checkpoint saved: stage "{name}" ({(time.time()-t0)/60:.1f} min)')
    return res

def cpu_state(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}


In [ ]:
def dirichlet_partition(indices,y,n_clients,alpha,seed,min_size=20,max_share=None):
    """Label-skewed Dirichlet split with a client-size cap.
    Classes are split with Dir(alpha) proportions exactly as before; if a client would exceed ceil(max_share * N) rows,
    its overflow is re-assigned to clients with spare capacity in proportion to the same Dirichlet weights, so label
    skew is preserved while no client dominates. With max_share >= 1 the random stream and the result are IDENTICAL to
    the uncapped split, so the dominant-client scenario reproduces that split exactly."""
    max_share=CONFIG.get('MAX_CLIENT_SHARE',1.0) if max_share is None else max_share
    rng=np.random.default_rng(seed); indices=np.array(indices); yy=y[indices]; N=len(indices)
    cap=N if max_share>=1 else max(int(np.ceil(max_share*N)),int(np.ceil(N/n_clients)))
    for _ in range(60):
        buckets=[[] for _ in range(n_clients)]; load=np.zeros(n_clients,dtype=int)
        for c in np.unique(yy):
            ids=indices[yy==c].copy(); rng.shuffle(ids); p=rng.dirichlet(np.full(n_clients,alpha)); cuts=(np.cumsum(p)*len(ids)).astype(int)[:-1]
            want=np.diff(np.concatenate([[0],cuts,[len(ids)]]))
            alloc=np.minimum(want,np.maximum(cap-load,0)); overflow=int(len(ids)-alloc.sum())
            while overflow>0:
                free=np.maximum(cap-load-alloc,0)
                if free.sum()==0: break
                w=p*(free>0); w=w/w.sum() if w.sum()>0 else (free>0)/(free>0).sum()
                extra=np.minimum(np.floor(w*overflow).astype(int),free)
                if extra.sum()==0:
                    j=int(np.argmax(np.where(free>0,w,-1))); extra[j]=min(int(free[j]),overflow)
                alloc=alloc+extra; overflow-=int(extra.sum())
            start=0
            for k in range(n_clients):
                buckets[k].extend(ids[start:start+alloc[k]].tolist()); start+=int(alloc[k])
            load=load+alloc
        if min(map(len,buckets))>=min_size: return [np.array(b,dtype=int) for b in buckets]
    buckets=[[] for _ in range(n_clients)]
    for j,i in enumerate(indices): buckets[j%n_clients].append(int(i))
    return [np.array(b,dtype=int) for b in buckets]

clients=dirichlet_partition(train_idx,y,CONFIG['N_CLIENTS'],CONFIG['DIRICHLET_ALPHA'],CONFIG['SEED'])
test_clients=dirichlet_partition(test_idx,y,CONFIG['N_CLIENTS'],CONFIG['DIRICHLET_ALPHA'],CONFIG['SEED']+99,min_size=8)
rng=np.random.default_rng(CONFIG['SEED']); resource=[]
for k,b in enumerate(clients):
    resource.append({'Client':k,'TrainRows':len(b),'Availability':float(rng.uniform(.60,.96)),'Reliability':float(rng.uniform(.72,1.0)),'MeanStaleness':float(rng.uniform(.2,2.0))})
resource_df=pd.DataFrame(resource); resource_df['TrainShare']=resource_df.TrainRows/resource_df.TrainRows.sum()
save_table(resource_df,'Table_03_Client_Resource_Profiles')
print(f"client training shares (cap {CONFIG['MAX_CLIENT_SHARE']}):",resource_df.TrainShare.round(3).tolist())
rows=[]
for k,b in enumerate(clients):
    counts=np.bincount(y[b],minlength=len(classes))
    for ci,c in enumerate(classes): rows.append({'Client':k,'Class':c,'Count':int(counts[ci]),'Fraction':counts[ci]/max(1,len(b))})
client_dist=pd.DataFrame(rows); save_table(client_dist,'Table_04_NonIID_Client_Distribution')
pivot=client_dist.pivot(index='Client',columns='Class',values='Fraction').fillna(0)
plt.figure(figsize=(max(7,len(classes)),4.5)); plt.imshow(pivot.values,aspect='auto'); plt.colorbar(label='class fraction'); plt.yticks(range(len(pivot)),pivot.index); plt.xticks(range(len(pivot.columns)),pivot.columns,rotation=45,ha='right'); plt.title('Non-IID class distribution across simulated farm gateways'); savefig('Figure_02_Client_Class_Heatmap')


In [ ]:
class LiteMLP(nn.Module):
    def __init__(self,n_features,n_classes,d=32,dropout=.1):
        super().__init__(); h=max(16,d)
        self.net=nn.Sequential(nn.Linear(n_features,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(dropout),nn.Linear(h,h),nn.GELU(),nn.Dropout(dropout),nn.Linear(h,n_classes))
    def forward(self,x): return self.net(x)

def new_model(seed=42):
    seed_all(seed); return LiteMLP(len(num),len(classes),CONFIG['D_MODEL'],CONFIG['DROPOUT']).to(DEVICE)
def count_params(model): return sum(p.numel() for p in model.parameters())
base_model=new_model(); print('Parameters:',count_params(base_model))


In [ ]:
def np_predict(model,indices,batch=2048):
    model.eval(); outs=[]
    with torch.no_grad():
        for s in range(0,len(indices),batch):
            xb=torch.tensor(X[indices[s:s+batch]],dtype=torch.float32,device=DEVICE); outs.append(torch.softmax(model(xb),1).cpu().numpy())
    return np.vstack(outs) if outs else np.empty((0,len(classes)))

def macro_fpr(ytrue,pred,nc):
    vals=[]
    for c in range(nc):
        yt=(ytrue==c); yp=(pred==c); fp=np.sum(~yt & yp); tn=np.sum(~yt & ~yp); vals.append(fp/max(1,fp+tn))
    return float(np.mean(vals))

def metric_row(model,indices,method,train_seconds=np.nan,comm_mb=np.nan):
    prob=np_predict(model,indices); yt=y[indices]; yp=prob.argmax(1)
    try: aucm=roc_auc_score(yt,prob,multi_class='ovr',average='macro',labels=np.arange(len(classes)))
    except Exception: aucm=np.nan
    p,r,f,_=precision_recall_fscore_support(yt,yp,average='macro',zero_division=0)
    return {'Method':method,'Accuracy':accuracy_score(yt,yp),'BalancedAccuracy':balanced_accuracy_score(yt,yp),'MacroPrecision':p,'MacroRecall':r,'MacroF1':f,'MacroAUROC':aucm,'MacroFPR':macro_fpr(yt,yp,len(classes)),'LogLoss':log_loss(yt,prob,labels=np.arange(len(classes))),'TrainSeconds':train_seconds,'CommMB':comm_mb,'Parameters':count_params(model)}

counts=np.bincount(y[train_idx],minlength=len(classes)).astype(float); class_w=counts.sum()/(len(classes)*np.maximum(counts,1)); class_w=torch.tensor(class_w,dtype=torch.float32,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=class_w)

def local_train(global_state,indices,seed,prox_mu=0.0,epochs=None,labels_override=None):
    epochs=CONFIG['LOCAL_EPOCHS'] if epochs is None else epochs; model=new_model(seed); model.load_state_dict(global_state)
    ref={k:v.detach().clone().to(DEVICE) for k,v in global_state.items()}; opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['LR'],weight_decay=CONFIG['WEIGHT_DECAY'])
    yy=y[indices] if labels_override is None else labels_override
    ds=TensorDataset(torch.tensor(X[indices],dtype=torch.float32),torch.tensor(yy,dtype=torch.long)); gen=torch.Generator(); gen.manual_seed(int(seed))
    dl=DataLoader(ds,batch_size=CONFIG['BATCH_SIZE'],shuffle=True,generator=gen)
    losses=[]; model.train()
    for _ in range(epochs):
        for xb,yb in dl:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); opt.zero_grad(); loss=criterion(model(xb),yb)
            if prox_mu>0:
                prox=sum(torch.sum((p-ref[name])**2) for name,p in model.named_parameters()); loss=loss+0.5*prox_mu*prox
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step(); losses.append(float(loss.detach().cpu()))
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()},float(np.mean(losses) if losses else np.nan)

def flatten_delta(local,base): return torch.cat([(local[k].float()-base[k].float()).reshape(-1) for k in base])
def state_from_delta(base,delta,template=None):
    out={}; off=0
    for k,v in base.items():
        n=v.numel(); out[k]=(v.float()+delta[off:off+n].reshape(v.shape)).type_as(v); off+=n
    return out


In [ ]:
def weighted_state_average(states,weights):
    w=np.asarray(weights,dtype=float); w=w/np.maximum(w.sum(),1e-12); out={}
    for k in states[0]: out[k]=sum(float(wi)*st[k].float() for wi,st in zip(w,states))
    return out

def coordinate_median(states):
    return {k:torch.stack([s[k].float() for s in states],0).median(0).values for k in states[0]}

def trimmed_mean(states,trim_frac=.2):
    n=len(states); trim=int(math.floor(trim_frac*n)); out={}
    for k in states[0]:
        z=torch.stack([s[k].float() for s in states],0); zs,_=torch.sort(z,dim=0); core=zs[trim:n-trim] if n-2*trim>=1 else zs; out[k]=core.mean(0)
    return out

def aggregate_multikrum(states,current,f=2,m=None):
    """Multi-Krum robust aggregation over flattened model deltas."""
    if len(states)<3:
        return weighted_state_average(states,[1.0]*len(states))
    V=np.vstack([flatten_delta(s,current).detach().cpu().numpy() for s in states])
    n=len(V); f=min(int(f),max(0,(n-3)//2)); neigh=max(1,n-f-2)
    dist=((V[:,None,:]-V[None,:,:])**2).sum(-1)
    scores=[]
    for i in range(n):
        ds=np.sort(np.delete(dist[i],i))
        scores.append(ds[:neigh].sum())
    if m is None: m=max(1,n-f-2)
    keep=np.argsort(scores)[:min(m,n)]
    return weighted_state_average([states[i] for i in keep],[1.0]*len(keep))

TRUST_COMPONENTS_FULL=frozenset({'direction','norm','reliability','history','clip','similarity','gate','probation'})

def own_deltas(states,current,bases=None):
    if bases is None: bases=[current]*len(states)
    return [flatten_delta(st,b) for st,b in zip(states,bases)],bases

def robust_update_features(states,current,history_trust=None,stales=None,reliabilities=None,bases=None,
                           components=TRUST_COMPONENTS_FULL,trust_update=None):
    vecs,bases=own_deltas(states,current,bases); n=len(vecs)
    norms=np.array([max(float(v.norm()),1e-12) for v in vecs])
    unit=torch.stack([v/(v.norm()+1e-12) for v in vecs]); logn=np.log(norms)
    #similarity signal: maximum cosine to any OTHER client's own update this round
    if n>1:
        S=np.asarray((unit@unit.T).detach().cpu().numpy() if hasattr(unit,'detach') else unit@unit.T,dtype=float)
        np.fill_diagonal(S,-1.0); maxsim=S.max(1)
    else:
        S=np.full((n,n),-1.0); maxsim=np.zeros(n)
    sim_raw=np.clip((CONFIG['SIM_FULL']-maxsim)/(CONFIG['SIM_FULL']-CONFIG['SIM_START']),0.0,1.0)
    sim_score=CONFIG['SIM_FLOOR']+(1-CONFIG['SIM_FLOOR'])*sim_raw
    if 'similarity' in components and n>1:
                parent=list(range(n))
        def _f(a):
            while parent[a]!=a: parent[a]=parent[parent[a]]; a=parent[a]
            return a
        for a_ in range(n):
            for b_ in range(a_+1,n):
                if S[a_,b_]>CONFIG['SIM_START']: parent[_f(a_)]=_f(b_)
        groups={}
        for a_ in range(n): groups.setdefault(_f(a_),[]).append(a_)
        reps=[unit[g].mean(0) if len(g)>1 else unit[g[0]] for g in groups.values()]
        ref=torch.stack(reps).median(0).values
        med=float(np.median([np.mean(logn[g]) for g in groups.values()]))
        n_groups=len(groups)
    else:
        ref=unit.median(0).values; med=float(np.median(logn)); n_groups=n
    ref=ref/(ref.norm()+1e-12)
    cos=np.array([float(torch.dot(u,ref)) for u in unit]); mad=np.median(np.abs(logn-med))+1e-6
    norm_score=np.exp(-np.abs(logn-med)/(2.5*mad)); dir_score=np.clip((cos+1)/2,0,1)
    history_trust=np.ones(n) if history_trust is None else np.asarray(history_trust,dtype=float)
    reliabilities=np.ones(n) if reliabilities is None else np.asarray(reliabilities,dtype=float)
    inst=np.ones(n)
    if 'direction' in components: inst=inst*dir_score**0.55
    if 'norm' in components: inst=inst*norm_score**0.30
    if 'reliability' in components: inst=inst*reliabilities**0.15
    if 'similarity' in components: inst=inst*sim_score
    inst=np.clip(inst,CONFIG['TRUST_FLOOR'],1.0)
    if 'history' in components:
       mode=trust_update or CONFIG.get('TRUST_UPDATE','symmetric')
        if mode=='asymmetric':
            rho=np.where(inst<history_trust-CONFIG['TRUST_DEADBAND'],CONFIG['TRUST_RHO_DOWN'],CONFIG['TRUST_RHO_UP'])
        else:
            rho=CONFIG['TRUST_RHO']
        upd=np.clip(rho*history_trust+(1-rho)*inst,CONFIG['TRUST_FLOOR'],1.0)
    else:
        upd=inst
    return pd.DataFrame({'DirectionScore':dir_score,'NormScore':norm_score,'MaxSimilarity':maxsim,'SimilarityScore':sim_score,
                         'ConsensusVotes':n_groups,'InstantTrust':inst,'UpdatedTrust':upd,'UpdateNorm':norms}),vecs

GATE_STATE={'median_cal':None}   # set by the warm-up calibration cell

def admission_gate(trusts,tau_abs,mode=None,kappa=None,median_cal=None):
    """Returns (admitted mask, effective threshold).
    hybrid   : drift-normalised threshold  tau_t = tau_cal * min(1, median_t(T) / median_cal(T)).
               The honest-majority median tracks benign trust drift late in training, so the calibrated
               benign/attacker separation is preserved without being widened by attackers themselves.
    absolute : tau_t = tau_cal (v2 behaviour, reported for comparison).
    relative : tau_t = median_t - kappa*1.4826*MAD_t (reported option; MAD is inflated by attackers).
    A minimum quorum of admitted clients is always enforced."""
    t=np.asarray(trusts,dtype=float); n=len(t)
    mode=CONFIG['GATE_MODE'] if mode is None else mode; kappa=CONFIG['GATE_KAPPA'] if kappa is None else kappa
    if mode=='off' or n==0: return np.ones(n,dtype=bool),0.0
    med=float(np.median(t))
    if mode=='absolute': eff=float(tau_abs)
    elif mode=='relative': eff=med-kappa*max(1.4826*float(np.median(np.abs(t-med))),CONFIG['GATE_MAD_FLOOR'])
    else:
        mc=(median_cal if median_cal is not None else GATE_STATE.get('median_cal')) or 1.0
        eff=float(tau_abs)*min(1.0,med/mc)
    keep=t>=eff
    q=max(1,int(math.ceil(CONFIG['MIN_QUORUM_FRAC']*n)))
    if keep.sum()<q:
        keep=np.zeros(n,dtype=bool); keep[np.argsort(t)[-q:]]=True
    return keep,float(eff)

def clip_own_deltas(states,bases,mult=2.5):
    vecs=[flatten_delta(st,b) for st,b in zip(states,bases)]; norms=np.array([float(v.norm()) for v in vecs]); cap=max(1e-12,mult*np.median(norms)); out=[]
    for b,v,n in zip(bases,vecs,norms):
        vv=v*(cap/max(n,cap)) if n>cap else v; out.append(state_from_delta(b,vv))
    return out,cap

# kept for backward compatibility with any user code that calls the v2 name
def clip_states_to_median_norm(states,current,mult=2.5): return clip_own_deltas(states,[current]*len(states),mult)

def aggregate_agritrust(states,current,ns,stales,reliabilities,history_trust,threshold,bases=None,
                        components=TRUST_COMPONENTS_FULL,gate_mode=None,kappa=None,trust_update=None,median_cal=None,
                        quarantine_streak=None):
    if bases is None: bases=[current]*len(states)
    feat,_=robust_update_features(states,current,history_trust,stales,reliabilities,bases,components,trust_update); trusts=feat.UpdatedTrust.values
    if 'clip' in components: clipped,cap=clip_own_deltas(states,bases,CONFIG['CLIP_MULTIPLIER'])
    else: clipped,cap=states,np.nan
    if 'gate' in components: keep,eff=admission_gate(trusts,threshold,gate_mode,kappa,median_cal)
    else: keep,eff=np.ones(len(states),dtype=bool),np.nan
        prob=np.zeros(len(states),dtype=bool)
    if 'gate' in components and 'probation' in components and quarantine_streak is not None:
        prob=(~keep)&(np.asarray(quarantine_streak)>=CONFIG['PROBATION_AFTER']); keep=keep|prob
    weights=[(n*math.exp(-CONFIG['STALENESS_LAMBDA']*s)*t*(CONFIG['PROBATION_WEIGHT'] if p else 1.0)) if k else 0.0
             for n,s,t,k,p in zip(ns,stales,trusts,keep,prob)]
    agg=weighted_state_average(clipped,weights)
    feat['Accepted']=keep; feat['Probation']=prob; feat['AggregationWeight']=weights; feat['ClipCap']=cap; feat['EffectiveThreshold']=eff
    return agg,feat

def pick_bad(ratio,seed):
    nb=max(1,int(round(ratio*len(clients)))); return {int(v) for v in np.random.default_rng(seed).choice(len(clients),size=nb,replace=False)}

def poison_states(states,bases,bad_pos,attack_type,seed,rnd=0):
    """Replace the OWN update of attacker positions in this round. bad_pos: positions in `states`."""
    rng=np.random.default_rng(seed); out=[]
    all_vec=[flatten_delta(s,b) for s,b in zip(states,bases)]
    benign_norms=[float(v.norm()) for j,v in enumerate(all_vec) if j not in bad_pos] or [float(v.norm()) for v in all_vec]
    med_norm=float(np.median(benign_norms))+1e-12
    bad_vec=[all_vec[i] for i in bad_pos]; collusive=None; alie=None
    if bad_vec:
        collusive=torch.stack(bad_vec).mean(0); collusive=collusive/(collusive.norm()+1e-12)
    for i,(st,b,v) in enumerate(zip(states,bases,all_vec)):
        if i not in bad_pos: out.append(st); continue
        if attack_type=='sign_flip': vv=-5.0*v
        elif attack_type=='model_replace': vv=8.0*v
        elif attack_type=='gaussian':
            vv=v+torch.tensor(rng.normal(0,1,size=v.shape),dtype=v.dtype)*(0.8*med_norm/math.sqrt(max(1,v.numel())))
        elif attack_type=='collusive_scale': vv=collusive*med_norm*float(CONFIG.get('COLLUSIVE_SCALE',6.0))
        elif attack_type=='adaptive_slow_drift':
            if alie is None:
                V=torch.stack(all_vec); z=min(CONFIG['DRIFT_Z_MAX'],CONFIG['DRIFT_Z0']+rnd*CONFIG['DRIFT_Z_STEP'])
                alie=V.mean(0)-z*V.std(0)
            vv=alie
        elif attack_type=='alie_jitter':

            if alie is None:
                V=torch.stack(all_vec); z=min(CONFIG['DRIFT_Z_MAX'],CONFIG['DRIFT_Z0']+rnd*CONFIG['DRIFT_Z_STEP'])
                alie=V.mean(0)-z*V.std(0)
            e=torch.tensor(rng.normal(0,1,size=tuple(alie.shape)),dtype=alie.dtype)
            vv=alie+e/(e.norm()+1e-12)*alie.norm()*float(CONFIG['ALIE_JITTER'])
        else: vv=v
        out.append(state_from_delta(b,vv))
    return out

def admission_stats(tl):
    """Client-round admission metrics from a trust log that carries Accepted and Malicious columns."""
    if tl is None or len(tl)==0 or 'Accepted' not in tl or tl['Accepted'].isna().all():
        return {'BenignFalseQuarantine':np.nan,'MaliciousQuarantineRecall':np.nan,'QuarantinePrecision':np.nan,'AdmittedFraction':np.nan,'MinAdmittedPerRound':np.nan,'MaxConsecutiveBenignQuarantine':np.nan,'ProbationAdmitsMalicious':np.nan,'ProbationAdmitsBenign':np.nan}
    acc=tl['Accepted'].astype(bool).values; mal=tl['Malicious'].astype(bool).values if 'Malicious' in tl else np.zeros(len(tl),dtype=bool)
    q=~acc
        mcq=0
    if 'Client' in tl:
        t2=tl.assign(_q=q,_m=mal).sort_values('Round')
        for _,g in t2[~t2._m].groupby('Client'):
            run=0
            for v in g._q.values:
                run=run+1 if v else 0; mcq=max(mcq,run)
    pr=tl['Probation'].fillna(False).astype(bool).values if 'Probation' in tl else np.zeros(len(tl),dtype=bool)
    return {'MaxConsecutiveBenignQuarantine':int(mcq),'ProbationAdmitsMalicious':int((pr&mal).sum()),'ProbationAdmitsBenign':int((pr&~mal).sum()),'BenignFalseQuarantine':float((q&~mal).sum()/max(1,(~mal).sum())),
            'MaliciousQuarantineRecall':float((q&mal).sum()/mal.sum()) if mal.sum() else np.nan,
            'QuarantinePrecision':float((q&mal).sum()/q.sum()) if q.sum() else np.nan,
            'AdmittedFraction':float(acc.mean()),
            'MinAdmittedPerRound':int(tl.assign(_a=acc).groupby('Round')._a.sum().min())}


RUN_REGISTRY=[]; CURRENT_STAGE='setup'

LAST_TRUST_STATE=None   # trust history + quarantine streaks at the end of the most recent federated run

PROGRESS={'stage':None,'n':0,'t0':None}
ROUND_SECONDS=[]

In [ ]:
# Cell 9 — Protocol-matched warm-up trust calibration and federated baselines
# tau_cal is calibrated on benign warm-up rounds of the SAME buffered staleness-aware protocol used later
# (dropout + staleness active, admission gate disabled). Assumption to report: warm-up updates are benign.

SYNC_METHODS=('FedAvg','FedProx','CoordMedian','TrimmedMean','MultiKrum')

def run_federated(method,seed=42,rounds=None,dropout_override=None,max_staleness_override=None,
                  init_state=None,attack=None,noise_sigma=0.0,threshold=None,gate_mode=None,kappa=None,
                  components=TRUST_COMPONENTS_FULL,trust_update=None,median_cal=None,init_trust=None):
    """attack: None or {'type':str,'ratio':float,'bad':set(client ids)} — attackers are persistent across rounds.
    noise_sigma: benign Gaussian noise added to every client's own update (robustness proxy, NOT differential privacy)."""
    rounds=CONFIG['ROUNDS'] if rounds is None else rounds; rng=np.random.default_rng(seed); model=new_model(seed)
    RUN_REGISTRY.append({'Stage':globals().get('CURRENT_STAGE','?'),'Method':method,'Rounds':rounds,
        'Init':'warm-start' if init_state is not None else 'scratch','GateMode':gate_mode or CONFIG['GATE_MODE'],
        'Threshold':round(float(threshold if threshold is not None else globals().get('trust_threshold',np.nan)),4),
        'Lambda':CONFIG['STALENESS_LAMBDA'],'Components':'+'.join(sorted(components)),'TrustUpdate':trust_update or CONFIG['TRUST_UPDATE'],
        'Attack':(f"{attack['type']}@{attack['ratio']}" if attack else 'none'),'InitTrust':'carried' if init_trust else 'fresh','NoiseSigma':noise_sigma,
        'DropoutOverride':dropout_override,'MaxStalenessOverride':max_staleness_override})
    if init_state is not None: model.load_state_dict(init_state)
    current={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    thr=trust_threshold if threshold is None else threshold
    bad=set() if attack is None else {int(b) for b in attack['bad']}
    hist_trust=np.ones(len(clients)); q_streak=np.zeros(len(clients),dtype=int)
    if init_trust is not None:
        hist_trust=np.asarray(init_trust['hist'],dtype=float).copy(); q_streak=np.asarray(init_trust['q_streak'],dtype=int).copy()
    history_states=[copy.deepcopy(current)]; rows=[]; trust_rows=[]; t0=time.time(); total_bytes=0
    param_bytes=sum(v.numel()*v.element_size() for v in current.values())
    for rnd in range(rounds):
        selected=[]
        for k in range(len(clients)):
            p=float(resource_df.loc[k,'Availability']) if dropout_override is None else 1-float(dropout_override)
            if method in SYNC_METHODS or rng.random()<p: selected.append(k)
        if len(selected)<2: selected=[int(v) for v in rng.choice(len(clients),size=min(2,len(clients)),replace=False)]
        states=[]; bases=[]; ns=[]; stales=[]; reli=[]; losses=[]
        for k in selected:
            mx=CONFIG['MAX_STALENESS'] if max_staleness_override is None else int(max_staleness_override)
            s=0 if method in SYNC_METHODS else int(min(mx,rng.poisson(resource_df.loc[k,'MeanStaleness'])))
            s=min(s,len(history_states)-1); start=history_states[len(history_states)-1-s]
            prox=CONFIG['FEDPROX_MU'] if method in ('FedProx','AgriTrust-ACFL') else 0.0
            st,loss=local_train(start,clients[k],seed+rnd*100+k,prox_mu=prox)
            states.append(st); bases.append(start); ns.append(len(clients[k])); stales.append(s); reli.append(float(resource_df.loc[k,'Reliability'])); losses.append(loss)
        if noise_sigma>0:
            vecs=[flatten_delta(st,b) for st,b in zip(states,bases)]; ref_scale=float(np.median([float(v.std()) for v in vecs]))+1e-12
            nrng=np.random.default_rng(seed*7+rnd)
            states=[state_from_delta(b,v+torch.tensor(nrng.normal(0,1,size=v.shape),dtype=v.dtype)*float(noise_sigma*ref_scale)) for b,v in zip(bases,vecs)]
        bad_pos={j for j,k in enumerate(selected) if int(k) in bad}
        if bad_pos: states=poison_states(states,bases,bad_pos,attack['type'],seed+rnd*31,rnd=rnd)
        if method=='CoordMedian': newstate=coordinate_median(states); feat=None
        elif method=='TrimmedMean': newstate=trimmed_mean(states,CONFIG['TRIM_FRAC']); feat=None
        elif method=='MultiKrum': newstate=aggregate_multikrum(states,current,CONFIG.get('MULTIKRUM_F',2)); feat=None
        elif method=='Async-Stale': newstate=weighted_state_average(states,[n*math.exp(-CONFIG['STALENESS_LAMBDA']*s) for n,s in zip(ns,stales)]); feat=None
        elif method=='RawTrust':
            feat,_=robust_update_features(states,current,hist_trust[selected],stales,reli,bases,trust_update=trust_update); tr=feat.UpdatedTrust.values
            newstate=weighted_state_average(states,[n*t for n,t in zip(ns,tr)]); hist_trust[selected]=tr
        elif method=='AgriTrust-ACFL':
            newstate,feat=aggregate_agritrust(states,current,ns,stales,reli,hist_trust[selected],thr,bases,components,gate_mode,kappa,trust_update,median_cal,q_streak[selected])
            hist_trust[selected]=feat.UpdatedTrust.values
            _acc=feat.Accepted.values.astype(bool); _pr=feat.Probation.values.astype(bool)
            q_streak[selected]=np.where(_acc&~_pr,0,np.where(_pr,0,q_streak[selected]+1))
        else: newstate=weighted_state_average(states,ns); feat=None
        current={k:v.detach().cpu().clone() for k,v in newstate.items()}; model.load_state_dict(current); history_states.append(copy.deepcopy(current))
        history_states=history_states[-(CONFIG['MAX_STALENESS']+2):]   # bounded buffer; enough for max staleness
        total_bytes += 2*param_bytes*len(selected)
        mr=metric_row(model,val_idx,method); rows.append({'Method':method,'Round':rnd+1,'SelectedClients':len(selected),'MeanStaleness':np.mean(stales),'ValMacroF1':mr['MacroF1'],'MeanLocalLoss':np.mean(losses)})
        if feat is not None:
            for j,k in enumerate(selected): trust_rows.append({'Method':method,'Round':rnd+1,'Client':int(k),'Staleness':stales[j],'Malicious':int(k) in bad,**feat.iloc[j].to_dict()})
    globals()['LAST_TRUST_STATE']={'hist':hist_trust.copy(),'q_streak':q_streak.copy()}
    ROUND_SECONDS.append((time.time()-t0)/max(1,rounds))
    _st=globals().get('CURRENT_STAGE','?')
    if PROGRESS['stage']!=_st: PROGRESS.update(stage=_st,n=0,t0=time.time())
    PROGRESS['n']+=1
    print(f"  [{_st}] run {PROGRESS['n']}: {method}"+(f" | {attack['type']}@{attack['ratio']}" if attack else '')
          +f" | {rounds} rounds, {time.time()-t0:.0f}s | stage elapsed {(time.time()-PROGRESS['t0'])/60:.1f} min",flush=True)
    return model,pd.DataFrame(rows),pd.DataFrame(trust_rows),time.time()-t0,total_bytes/1024**2

def warm_start(method,seed,rounds=None,**kw):
    """ a method's OWN clean warm-up. Returns its model state, its trust state and its validation Macro-F1."""
    rounds=CONFIG['WARM_START_ROUNDS'] if rounds is None else rounds
    m,_,_,_,_=run_federated(method,seed,rounds=rounds,**kw)
    return {'state':{k:v.detach().cpu().clone() for k,v in m.state_dict().items()},'trust':copy.deepcopy(LAST_TRUST_STATE),
            'val_f1':metric_row(m,val_idx,method)['MacroF1']}

def selection_runs(seed_base):
    """ selection protocol (validation only), averaged over SELECT_SEEDS_N seeds: a clean run from scratch, then each
    attack scenario as a CONTINUATION of that clean run (trust state carried).  attacked from round 1, where every
    candidate collapsed to the same score (0.344), so the choice was decided by noise."""
    rows=[]
    for k in range(CONFIG['SELECT_SEEDS_N']):
        sd=seed_base+101*k
        mdl,_,tl,_,_=run_federated('AgriTrust-ACFL',sd,rounds=CONFIG['LAMBDA_SELECT_ROUNDS'])
        st={kk:v.detach().cpu().clone() for kk,v in mdl.state_dict().items()}; tr=copy.deepcopy(LAST_TRUST_STATE)
        rows.append({'Seed':sd,'Scenario':'clean','ValMacroF1':metric_row(mdl,val_idx,'AgriTrust-ACFL')['MacroF1'],**admission_stats(tl)})
        for si,(at,ratio) in enumerate([x for x in CONFIG['LAMBDA_SELECT_SCENARIOS'] if x[0]!='none']):
            attack={'type':at,'ratio':ratio,'bad':pick_bad(ratio,sd+7+si)}
            m2,_,tl2,_,_=run_federated('AgriTrust-ACFL',sd+1+si,rounds=CONFIG['POISON_ROUNDS'],init_state=st,init_trust=tr,attack=attack)
            rows.append({'Seed':sd,'Scenario':f'{at}@{ratio} after clean warm-up','ValMacroF1':metric_row(m2,val_idx,'AgriTrust-ACFL')['MacroF1'],**admission_stats(tl2)})
    return rows

# ---- protocol-matched warm-up calibration (gate disabled, benign, dropout + staleness active)
CURRENT_STAGE='calibration'
def calibrate_trust(mode,seed):
    """Benign warm-up of the real protocol under ONE trust rule. The first TRUST_CAL_BURNIN rounds are excluded:
    trust starts at 1.0 and is still falling there, which would inflate tau."""
    _,_,cl,_,_=run_federated('AgriTrust-ACFL',seed,rounds=CONFIG['TRUST_WARMUP_ROUNDS'],threshold=0.0,gate_mode='off',trust_update=mode)
    burn=min(CONFIG['TRUST_CAL_BURNIN'],max(0,CONFIG['TRUST_WARMUP_ROUNDS']-1))
    vals=cl.loc[cl.Round>burn,'UpdatedTrust'].values if len(cl) else np.array([])
    tau=float(np.quantile(vals,CONFIG['TRUST_ALPHA'])) if len(vals)>=4 else CONFIG['DEFAULT_TRUST_THRESHOLD']
    tau=float(np.clip(tau,CONFIG['TRUST_FLOOR'],CONFIG['TRUST_THRESHOLD_CAP']))
    return {'tau':tau,'median':float(np.median(vals)) if len(vals) else None,'n':int(len(vals)),'burnin':burn,
            'log':cl.assign(TrustUpdate=mode,UsedForCalibration=cl.Round>burn) if len(cl) else cl}

# Each candidate rule gets its OWN tau and median: a tau calibrated under one rule is wrong for the other.
TRUST_MODES=(['symmetric','asymmetric'] if CONFIG.get('SELECT_TRUST_UPDATE_ON_VALIDATION',True) else [CONFIG['TRUST_UPDATE']])
TRUST_CAL={}

def use_trust_mode(m):
    global trust_threshold
    CONFIG['TRUST_UPDATE']=m; trust_threshold=TRUST_CAL[m]['tau']; GATE_STATE['median_cal']=TRUST_CAL[m]['median']

def _trust_setup():
    """Calibration of every candidate rule + validation choice of the rule (one resumable stage).
    Choose on VALIDATION: mean validation Macro-F1 among rules whose mean clean benign false quarantine
    <= MAX_BENIGN_FQ_FOR_SELECTION (F1 alone could pick a rule that quarantines many honest gateways)."""
    global CURRENT_STAGE
    CURRENT_STAGE='calibration'
    TRUST_CAL.clear(); TRUST_CAL.update({m:calibrate_trust(m,CONFIG['SEED']+777) for m in TRUST_MODES})
    CURRENT_STAGE='trust_select'; tsel=None; chosen=TRUST_MODES[0]
    if len(TRUST_MODES)>1:
        trows=[]
        for m in TRUST_MODES:
            use_trust_mode(m)
            for r in selection_runs(CONFIG['SEED']+333): trows.append({'TrustUpdate':m,'Tau_cal':TRUST_CAL[m]['tau'],**r})
        tsel=pd.DataFrame(trows)
        tsum=tsel.groupby('TrustUpdate').ValMacroF1.agg(['mean','std']).rename(columns={'mean':'MeanValMacroF1','std':'SD'})
        tsum['CleanFalseQuarantine']=tsel[tsel.Scenario=='clean'].groupby('TrustUpdate').BenignFalseQuarantine.mean()
        ok=tsum[tsum.CleanFalseQuarantine<=CONFIG['MAX_BENIGN_FQ_FOR_SELECTION']]
        chosen=ok.MeanValMacroF1.idxmax() if len(ok) else tsum.CleanFalseQuarantine.idxmin()
        tsel['Selected']=tsel.TrustUpdate==chosen
        print(tsum.round(4).to_string()); print('trust-memory rule selected on validation:',chosen)
    return {'TRUST_CAL':dict(TRUST_CAL),'tsel':tsel,'chosen':chosen}

_ts=run_stage('trust_setup',_trust_setup)
TRUST_CAL.clear(); TRUST_CAL.update(_ts['TRUST_CAL']); tsel=_ts['tsel']; chosen=_ts['chosen']
save_table(pd.concat([v['log'] for v in TRUST_CAL.values()],ignore_index=True),'Table_05b_Warmup_Trust_Log')
if tsel is not None: save_table(tsel,'Table_05e_Trust_Update_Selection_Validation')
CURRENT_STAGE='trust_select'
use_trust_mode(chosen)
save_table(pd.DataFrame([{'TrustUpdate':chosen,'TrustAlpha':CONFIG['TRUST_ALPHA'],'Tau_cal':trust_threshold,'Median_cal':GATE_STATE['median_cal'],
    'WarmupRounds':CONFIG['TRUST_WARMUP_ROUNDS'],'BurnInRoundsExcluded':TRUST_CAL[chosen]['burnin'],
    'WarmupClientRoundsUsed':TRUST_CAL[chosen]['n'],'GateMode':CONFIG['GATE_MODE'],'MinQuorumFrac':CONFIG['MIN_QUORUM_FRAC'],
    'RhoDown':CONFIG['TRUST_RHO_DOWN'],'RhoUp':CONFIG['TRUST_RHO_UP'],'Deadband':CONFIG['TRUST_DEADBAND'],'RhoSymmetric':CONFIG['TRUST_RHO'],
    'Protocol':'benign warm-up rounds of the buffered staleness-aware protocol; dropout and staleness active; admission gate disabled; burn-in rounds excluded',
    'SimStart':CONFIG['SIM_START'],'SimFull':CONFIG['SIM_FULL'],
    'HonestMaxSimilarityP99':float(TRUST_CAL[chosen]['log'].loc[TRUST_CAL[chosen]['log'].UsedForCalibration,'MaxSimilarity'].quantile(.99)) if 'MaxSimilarity' in TRUST_CAL[chosen]['log'] else np.nan,
    'Assumption':'warm-up client updates are benign'}]),'Table_05_Trust_Calibration')
print('Calibrated tau_cal =',round(trust_threshold,4),'| rule =',chosen,'| client-rounds used =',TRUST_CAL[chosen]['n'])

CURRENT_STAGE='lambda_select'; LAMBDA_DEFAULT=CONFIG['STALENESS_LAMBDA']
if CONFIG.get('SELECT_LAMBDA_ON_VALIDATION',True):
    lrows=[]
    for lam in CONFIG['LAMBDA_SENSITIVITY']:
        CONFIG['STALENESS_LAMBDA']=float(lam)
        for r in selection_runs(CONFIG['SEED']+333): lrows.append({'Lambda':float(lam),**r})
    lam_sel=pd.DataFrame(lrows); score=lam_sel.groupby('Lambda').ValMacroF1.mean()
    print('lambda selection (mean +/- SD over seeds and scenarios):',
          {k:f"{v['mean']:.4f}+/-{v['std']:.4f}" for k,v in lam_sel.groupby('Lambda').ValMacroF1.agg(['mean','std']).to_dict('index').items()})
    CONFIG['STALENESS_LAMBDA']=float(score.idxmax()); lam_sel['Selected']=lam_sel.Lambda==CONFIG['STALENESS_LAMBDA']
    save_table(lam_sel,'Table_05c_Lambda_Selection_Validation')
    print('lambda selected on validation =',CONFIG['STALENESS_LAMBDA'],'| mean val Macro-F1 by lambda:',score.round(4).to_dict())
else:
    print('lambda fixed a priori =',CONFIG['STALENESS_LAMBDA'])

CURRENT_STAGE='main'
methods=['FedAvg','FedProx','Async-Stale','CoordMedian','TrimmedMean','MultiKrum','RawTrust','AgriTrust-ACFL']

def _main_runs():
    out={'states':{},'HIST':{},'TRUSTLOG':[],'PERF':[]}
    for i,m in enumerate(methods):
        mdl,h,tlog,secs,comm=run_federated(m,CONFIG['SEED']+i*100)
        out['states'][m]=cpu_state(mdl); out['HIST'][m]=h; out['PERF'].append(metric_row(mdl,test_idx,m,secs,comm))
        if len(tlog): out['TRUSTLOG'].append(tlog)
    return out

_mr=run_stage('main_runs',_main_runs)
MODELS={}
for m,st in _mr['states'].items():
    MODELS[m]=new_model(CONFIG['SEED']); MODELS[m].load_state_dict(st); MODELS[m].eval()
HIST=_mr['HIST']; TRUSTLOG=_mr['TRUSTLOG']; PERF=_mr['PERF']
perf=pd.DataFrame(PERF); save_table(perf,'Table_06_Main_Test_Performance'); hist=pd.concat(HIST.values(),ignore_index=True); save_table(hist,'Table_07_Training_History')
trustlog=pd.concat(TRUSTLOG,ignore_index=True) if TRUSTLOG else pd.DataFrame(); save_table(trustlog,'Table_08_Clean_Run_Trust_Log')
clean_adm=[]
if len(trustlog):
    for m,g in trustlog.groupby('Method'):
        a=admission_stats(g); clean_adm.append({'Method':m,'ClientRounds':len(g),'MeanTrust':g.UpdatedTrust.mean(),**a})
clean_admission=pd.DataFrame(clean_adm); save_table(clean_admission,'Table_08b_Clean_Run_Admission_Audit')
print(perf.sort_values('MacroF1',ascending=False).round(4).to_string(index=False))
print(clean_admission.round(4).to_string(index=False))

def _planned_rounds():
    R=CONFIG['ROUNDS']; W=CONFIG['WARM_START_ROUNDS']; P=CONFIG['POISON_ROUNDS']; na=len(CONFIG['POISON_TYPES'])*len(CONFIG['POISON_RATIOS'])
    ntau=len(CONFIG['TAU_SENSITIVITY'])+1; nsc=len(CONFIG['SENS_SCENARIOS'])
    plan={'warm starts + poisoning + cold start':7*W+7*P+na*7*P+7*max(P,R//3),
          'benign false-quarantine audit':2*len(CONFIG['NOISE_LEVELS'])*len(CONFIG['ROBUSTNESS_STALENESS'])*CONFIG['AUDIT_ROUNDS'],
          'connectivity grid':len(CONFIG['ROBUSTNESS_DROPOUT'])*len(CONFIG['ROBUSTNESS_STALENESS'])*2*max(2,R//2),
          'dominant-client scenario':(CONFIG['TRUST_WARMUP_ROUNDS']+8*R) if CONFIG.get('MAX_CLIENT_SHARE',1)<1 else 0,
          'ablation':CONFIG.get('ABLATION_SEEDS_N',1)*len(CONFIG['ABLATION_SCENARIOS'])*9*CONFIG['ABLATION_ROUNDS'],
          'sensitivity':(2*ntau+len(CONFIG['LAMBDA_SENSITIVITY']))*nsc*CONFIG['SENS_ROUNDS'],
          'multi-seed (clean + poisoning)':(len(CONFIG['PAPER_SEEDS'])*8*R+CONFIG['POISON_SEEDS_N']*(7*W+len(CONFIG['MULTISEED_POISON_SCENARIOS'])*7*P)) if CONFIG['RUN_MULTI_SEED'] else 0,
          'UNSW-NB15 federated replication':(CONFIG['TRUST_WARMUP_ROUNDS']+7*W+8*CONFIG['UNSW_FED_ROUNDS']+len(CONFIG['UNSW_POISON_SCENARIOS'])*6*P) if CONFIG.get('RUN_EXTERNAL_UNSW') else 0}
    return plan
_spr=float(np.median(ROUND_SECONDS)) if ROUND_SECONDS else np.nan
_plan=pd.DataFrame([{'Stage':k,'PlannedRounds':v,'EstimatedHours':v*_spr/3600 if not np.isnan(_spr) else np.nan,
                     'AlreadyCheckpointed':False} for k,v in _planned_rounds().items()])
_plan.loc[len(_plan)]={'Stage':'TOTAL (remaining stages)','PlannedRounds':_plan.PlannedRounds.sum(),'EstimatedHours':_plan.EstimatedHours.sum(),'AlreadyCheckpointed':False}
save_table(_plan,'Table_00d_Runtime_Plan')
if np.isnan(_spr):
    print('Runtime plan: all stages so far were loaded from checkpoints; timing will be measured by the next computed stage.')
else:
    print(f'Runtime plan (measured {_spr:.2f} s per round; rough, sync methods are slower than async ones):')
    print(_plan.round(2).to_string(index=False))
    if _plan.EstimatedHours.iloc[-1]>11:
        print('>>> Expect SEVERAL Colab sessions. After a disconnect: reconnect -> Run all. Finished stages load from checkpoints.')


## Model validation outputs
Held-out test diagnostics: class-wise performance, normalized confusion matrix, one-vs-rest ROC/PR, and confidence/error audit.


In [ ]:
# Model-validation tables and graphs
best_method=perf.sort_values('MacroF1',ascending=False).iloc[0].Method
best_model=MODELS[best_method]
prob=np_predict(best_model,test_idx); yt=y[test_idx]; yp=prob.argmax(1)

pc,rc,fc,sup=precision_recall_fscore_support(yt,yp,labels=np.arange(len(classes)),zero_division=0)
class_rows=[]
for cid,cname in enumerate(classes):
    try: au=roc_auc_score((yt==cid).astype(int),prob[:,cid])
    except Exception: au=np.nan
    try: ap=average_precision_score((yt==cid).astype(int),prob[:,cid])
    except Exception: ap=np.nan
    class_rows.append({'Class':cname,'Support':int(sup[cid]),'Precision':pc[cid],'Recall':rc[cid],
                       'F1':fc[cid],'AUROC':au,'AUPRC':ap})
class_perf=pd.DataFrame(class_rows)
save_table(class_perf,'Table_06b_Classwise_Test_Performance')

cm=confusion_matrix(yt,yp,labels=np.arange(len(classes)),normalize='true')
pd.DataFrame(cm,index=classes,columns=classes).to_csv(TAB/'Table_06c_Normalized_Confusion_Matrix.csv')
plt.figure(figsize=(8,6)); plt.imshow(cm,aspect='auto',vmin=0,vmax=1); plt.colorbar(label='Row-normalized proportion')
plt.xticks(range(len(classes)),classes,rotation=45,ha='right'); plt.yticks(range(len(classes)),classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Normalized confusion matrix — {best_method}')
savefig('Figure_05_Normalized_Confusion_Matrix')

plt.figure(figsize=(7,5)); roc_rows=[]
for cid,cname in enumerate(classes):
    yy=(yt==cid).astype(int)
    if yy.min()==yy.max(): continue
    fpr,tpr,_=roc_curve(yy,prob[:,cid]); a=auc(fpr,tpr)
    plt.plot(fpr,tpr,label=f'{cname} ({a:.3f})'); roc_rows.append({'Class':cname,'AUROC':a})
plt.plot([0,1],[0,1],linestyle='--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(fontsize=7)
plt.title(f'Class-wise ROC — {best_method}'); savefig('Figure_06_Classwise_ROC')
save_table(pd.DataFrame(roc_rows),'Table_06d_Classwise_ROC_AUC')

plt.figure(figsize=(7,5)); pr_rows=[]
for cid,cname in enumerate(classes):
    yy=(yt==cid).astype(int)
    if yy.min()==yy.max(): continue
    prec,rec,_=precision_recall_curve(yy,prob[:,cid]); ap=average_precision_score(yy,prob[:,cid])
    plt.plot(rec,prec,label=f'{cname} ({ap:.3f})'); pr_rows.append({'Class':cname,'AUPRC':ap})
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(fontsize=7)
plt.title(f'Class-wise precision-recall — {best_method}'); savefig('Figure_06b_Classwise_PR')
save_table(pd.DataFrame(pr_rows),'Table_06e_Classwise_PR_AUC')

conf=prob.max(1)
val_audit=pd.DataFrame([{
    'BestMethod':best_method,'TestRows':len(yt),'Accuracy':accuracy_score(yt,yp),
    'MacroF1':f1_score(yt,yp,average='macro',zero_division=0),
    'MeanConfidenceCorrect':float(conf[yp==yt].mean()) if np.any(yp==yt) else np.nan,
    'MeanConfidenceWrong':float(conf[yp!=yt].mean()) if np.any(yp!=yt) else np.nan,
    'ErrorRate':float(np.mean(yp!=yt))
}])
save_table(val_audit,'Table_06f_Model_Validation_Audit')

plt.figure(figsize=(7,4))
plt.hist(conf[yp==yt],bins=20,alpha=.55,label='Correct')
plt.hist(conf[yp!=yt],bins=20,alpha=.55,label='Wrong')
plt.xlabel('Maximum predicted probability'); plt.ylabel('Count'); plt.legend()
plt.title(f'Prediction confidence audit — {best_method}')
savefig('Figure_06c_Confidence_Audit')


In [ ]:
# Cell 10 — Main performance, convergence, confusion matrix and ROC figures
for met in ['MacroF1','MacroAUROC','BalancedAccuracy']:
    plt.figure(figsize=(8,4)); plt.bar(perf.Method,perf[met]); plt.ylim(0,1.02); plt.ylabel(met); plt.xticks(rotation=25,ha='right'); plt.title(f'Test {met} comparison'); savefig(f'Figure_04_{met}_Comparison')
plt.figure(figsize=(8,4.8))
for m,h in HIST.items(): plt.plot(h.Round,h.ValMacroF1,marker='o',label=m)
plt.xlabel('Federated round'); plt.ylabel('Validation macro-F1'); plt.legend(fontsize=7,ncol=2); plt.title('Federated convergence under heterogeneous clients'); savefig('Figure_03_Training_Convergence')
best=MODELS['AgriTrust-ACFL']; prob=np_predict(best,test_idx); yt=y[test_idx]; yp=prob.argmax(1); cm=confusion_matrix(yt,yp,labels=np.arange(len(classes)),normalize='true')
plt.figure(figsize=(7,6)); plt.imshow(cm,vmin=0,vmax=1); plt.colorbar(label='row-normalized fraction'); plt.xticks(range(len(classes)),classes,rotation=45,ha='right'); plt.yticks(range(len(classes)),classes); plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('AgriTrust-ACFL normalized confusion matrix'); savefig('Figure_05_Confusion_Matrix')
plt.figure(figsize=(7,5))
for ci,c in enumerate(classes):
    truth=(yt==ci).astype(int)
    if truth.min()==truth.max(): continue
    fpr,tpr,_=roc_curve(truth,prob[:,ci]); plt.plot(fpr,tpr,label=f'{c} AUC={auc(fpr,tpr):.3f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.legend(fontsize=7); plt.title('One-vs-rest ROC curves'); savefig('Figure_06_ROC_Curves')


In [ ]:
POISON_METHODS=['FedAvg','CoordMedian','TrimmedMean','MultiKrum','Async-Stale','RawTrust','AgriTrust-ACFL']

def method_warm_starts(seed,methods_):
    return {m:warm_start(m,seed) for m in methods_}

WARM_STATES=run_stage('warm_states',lambda: method_warm_starts(CONFIG['SEED']+4242,POISON_METHODS))
save_table(pd.DataFrame([{'Method':m,'WarmStartRounds':CONFIG['WARM_START_ROUNDS'],'WarmValMacroF1':w['val_f1']} for m,w in WARM_STATES.items()]),
           'Table_09a_Per_Method_Warm_Start_Checkpoints')
# AgriTrust-only stress tests (benign audit, sensitivity, ablation) start from AgriTrust's own warm-up checkpoint
WARM_STATE=WARM_STATES['AgriTrust-ACFL']['state']; WARM_TRUST=WARM_STATES['AgriTrust-ACFL']['trust']

def poison_experiment(seed=2000,warm_states=None,ratios=None,types=None,methods_=None,rounds=None,eval_idx=None,include_none=True):
    ratios=CONFIG['POISON_RATIOS'] if ratios is None else ratios; types=CONFIG['POISON_TYPES'] if types is None else types
    methods_=POISON_METHODS if methods_ is None else methods_; rounds=CONFIG['POISON_ROUNDS'] if rounds is None else rounds
    eval_idx=test_idx if eval_idx is None else eval_idx; rows=[]; tlogs=[]
    warm_states=method_warm_starts(seed+4242,methods_) if warm_states is None else warm_states
    if include_none:   # no-attack reference: NO attackers at all (pick_bad would always pick >= 1 client)
        for meth in methods_:
            w=warm_states[meth]
            mdl,_,tl,secs,comm=run_federated(meth,seed+1,rounds=rounds,init_state=w['state'],init_trust=w['trust'])
            r=metric_row(mdl,eval_idx,meth,secs,comm); r.update({'PoisonRatio':0.0,'AttackType':'none','Attackers':''}); rows.append(r)
            if len(tl): tl=tl.copy(); tl['PoisonRatio']=0.0; tl['AttackType']='none'; tlogs.append(tl)
    for ratio in ratios:
        bad=pick_bad(ratio,seed+int(round(ratio*1000)))
        for ai,at in enumerate(types):
            sc_seed=seed+1   # ONE selection/staleness stream for every method AND scenario, incl. the no-attack
                             # reference, so AttackDamage is paired (attributable to the attack, not to stream noise)
            for meth in methods_:
                w=warm_states[meth]
                mdl,_,tl,secs,comm=run_federated(meth,sc_seed,rounds=rounds,init_state=w['state'],init_trust=w['trust'],
                                                 attack={'type':at,'ratio':ratio,'bad':bad})
                r=metric_row(mdl,eval_idx,meth,secs,comm); r.update({'PoisonRatio':ratio,'AttackType':at,'Attackers':','.join(map(str,sorted(bad)))}); rows.append(r)
                if len(tl): tl=tl.copy(); tl['PoisonRatio']=ratio; tl['AttackType']=at; tlogs.append(tl)
    df=pd.DataFrame(rows)
    if include_none and len(df):
        ref=df[df.AttackType=='none'].set_index('Method').MacroF1
        df['CleanContinuationMacroF1']=df.Method.map(ref); df['AttackDamage']=df.CleanContinuationMacroF1-df.MacroF1
    return df,(pd.concat(tlogs,ignore_index=True) if tlogs else pd.DataFrame())

poison_df,poison_trust=run_stage('poisoning',lambda: poison_experiment(warm_states=WARM_STATES))
save_table(poison_df,'Table_09_Poisoning_Robustness'); save_table(poison_trust,'Table_10_Poisoning_Client_Trust')

td=[]
for (meth,ratio,at),g in poison_trust.groupby(['Method','PoisonRatio','AttackType']):
    yy=g.Malicious.astype(int).values; score=1-g.UpdatedTrust.values
    try: au=roc_auc_score(yy,score); ap=average_precision_score(yy,score)
    except Exception: au=ap=np.nan
    last=g[g.Round==g.Round.max()]
    try: au_last=roc_auc_score(last.Malicious.astype(int).values,1-last.UpdatedTrust.values)
    except Exception: au_last=np.nan
    td.append({'Method':meth,'PoisonRatio':ratio,'AttackType':at,'ClientRounds':len(g),
               'TrustAUROC_ClientRound':au,'TrustAUPRC_ClientRound':ap,'TrustAUROC_FinalRound':au_last,
               'MeanTrustMalicious':g[g.Malicious].UpdatedTrust.mean(),'MeanTrustBenign':g[~g.Malicious].UpdatedTrust.mean(),
               **admission_stats(g)})
trust_detection=pd.DataFrame(td); save_table(trust_detection,'Table_11_Trust_Detection_and_False_Quarantine')

# Attack damage relative to each method's own no-attack continuation
_dm=poison_df[poison_df.AttackType!='none']
if 'AttackDamage' in _dm and len(_dm):
    piv=_dm[_dm.PoisonRatio==_dm.PoisonRatio.max()].pivot_table(index='AttackType',columns='Method',values='AttackDamage')
    ax=piv.plot(kind='bar',figsize=(10,4.5)); ax.axhline(0,color='k',linewidth=.8); ax.set_ylabel('Macro-F1 lost vs own clean continuation')
    plt.xticks(rotation=20,ha='right'); plt.legend(fontsize=7,ncol=2); plt.title(f'Attack damage at ratio {_dm.PoisonRatio.max()}'); savefig('Figure_07b_Attack_Damage')

# Limitation evidence: attackers present from round 1 (no clean warm-up, no trust history)
def _cold_start():
    cs=[]; _bad=pick_bad(0.30,CONFIG['SEED']+5151)
    for meth in POISON_METHODS:
        mdl,_,tl,secs,comm=run_federated(meth,CONFIG['SEED']+5151,rounds=max(CONFIG['POISON_ROUNDS'],CONFIG['ROUNDS']//3),
                                         attack={'type':'sign_flip','ratio':0.30,'bad':_bad})
        r=metric_row(mdl,test_idx,meth,secs,comm); r.update({'Scenario':'sign_flip@0.3 from round 1 (cold start)'})
        if meth=='AgriTrust-ACFL': r.update(admission_stats(tl))
        cs.append(r)
    return cs
cs=run_stage('cold_start',_cold_start)
save_table(pd.DataFrame(cs),'Table_09c_Cold_Start_Attack_Limitation')

plt.figure(figsize=(8,4.5)); gp=poison_df.groupby(['Method','PoisonRatio']).MacroF1.mean().reset_index()
for m,g in gp.groupby('Method'): plt.plot(g.PoisonRatio,g.MacroF1,marker='o',label=m)
plt.xlabel('Malicious-client ratio'); plt.ylabel('Macro-F1 (mean over attack types)'); plt.ylim(0,1.02); plt.legend(fontsize=7,ncol=2); plt.title('Poisoning robustness across malicious-client ratios'); savefig('Figure_07_Poisoning_Robustness')

pa=poison_trust[poison_trust.Method=='AgriTrust-ACFL']
if len(pa):
    at0='sign_flip' if 'sign_flip' in set(pa.AttackType) else pa.AttackType.iloc[0]
    g=pa[(pa.PoisonRatio==pa.PoisonRatio.max())&(pa.AttackType==at0)]
    plt.figure(figsize=(8,4.5))
    for c,h in g.groupby('Client'):
        mal=bool(h.Malicious.iloc[0]); plt.plot(h.Round,h.UpdatedTrust,marker='o',linestyle='--' if mal else '-',label=f'C{c}{" (mal)" if mal else ""}')
    eff=g.groupby('Round').EffectiveThreshold.first(); plt.step(eff.index,eff.values,where='mid',color='k',linewidth=1,label='effective gate')
    plt.xlabel('Round'); plt.ylabel('Updated trust'); plt.ylim(0,1.05); plt.legend(fontsize=6,ncol=3)
    plt.title(f'AgriTrust client trust under {at0} (ratio={pa.PoisonRatio.max()})'); savefig('Figure_08_Client_Trust_Under_Poisoning')


In [ ]:
CURRENT_STAGE='benign_audit'
def benign_noise_audit(seed=3100):
    rows=[]
    for mode in ['absolute','hybrid']:
        for sigma in CONFIG['NOISE_LEVELS']:
            for stmax in CONFIG['ROBUSTNESS_STALENESS']:
                mdl,_,tl,_,_=run_federated('AgriTrust-ACFL',seed,rounds=CONFIG['AUDIT_ROUNDS'],init_state=WARM_STATE,init_trust=WARM_TRUST,
                                           noise_sigma=float(sigma),max_staleness_override=stmax,gate_mode=mode)
                a=admission_stats(tl)
                rows.append({'GateMode':mode,'NoiseSigmaProxy':sigma,'MaxStaleness':stmax,'ClientRounds':len(tl),
                             'MeanTrust':tl.UpdatedTrust.mean(),'MinTrust':tl.UpdatedTrust.min(),
                             'FalseQuarantineRate':a['BenignFalseQuarantine'],'MinAdmittedPerRound':a['MinAdmittedPerRound'],
                             'Tau_cal':trust_threshold,'MeanEffectiveThreshold':tl.EffectiveThreshold.mean(),
                             'ValMacroF1':metric_row(mdl,val_idx,'AgriTrust-ACFL')['MacroF1']})
    return pd.DataFrame(rows)

noise_audit=run_stage('benign_audit',benign_noise_audit); save_table(noise_audit,'Table_12_Benign_Noise_Staleness_Trust_Audit')
fig,axs=plt.subplots(1,2,figsize=(11,4.2),sharey=True)
for ax,mode in zip(axs,['absolute','hybrid']):
    for st,g in noise_audit[noise_audit.GateMode==mode].groupby('MaxStaleness'): ax.plot(g.NoiseSigmaProxy,g.FalseQuarantineRate,marker='o',label=f'max stale={st}')
    ax.set_title(f'{mode} gate'); ax.set_xlabel('Gaussian update-noise proxy level'); ax.set_ylim(0,1.02)
axs[0].set_ylabel('Benign false-quarantine rate'); axs[1].legend(fontsize=7)
plt.suptitle('Benign false quarantine under update noise and staleness'); savefig('Figure_09_False_Quarantine_Noise_Staleness')


In [ ]:
CURRENT_STAGE='connectivity'
def _connectivity():
    conn=[]
    for d in CONFIG['ROBUSTNESS_DROPOUT']:
        for s in CONFIG['ROBUSTNESS_STALENESS']:
            for meth in ['Async-Stale','AgriTrust-ACFL']:
                mdl,_,_,secs,comm=run_federated(meth,5000+int(d*100)+s,rounds=1 if SMOKE else max(2,CONFIG['ROUNDS']//2),dropout_override=d,max_staleness_override=s)
                r=metric_row(mdl,test_idx,meth,secs,comm); r.update({'DropoutRate':d,'MaxStaleness':s}); conn.append(r)
    return conn
conn=run_stage('connectivity',_connectivity)
conn_df=pd.DataFrame(conn); save_table(conn_df,'Table_13_Connectivity_Robustness')
for meth in ['Async-Stale','AgriTrust-ACFL']:
    q=conn_df[conn_df.Method==meth].pivot(index='DropoutRate',columns='MaxStaleness',values='MacroF1')
    plt.figure(figsize=(6.5,4.5)); plt.imshow(q.values,aspect='auto',vmin=max(0,q.values.min()-.05),vmax=min(1,q.values.max()+.02)); plt.colorbar(label='Macro-F1'); plt.xticks(range(len(q.columns)),q.columns); plt.yticks(range(len(q.index)),q.index); plt.xlabel('Max staleness'); plt.ylabel('Dropout rate'); plt.title(f'{meth}: connectivity robustness'); savefig(f'Figure_10_{meth}_Connectivity_Heatmap')


In [ ]:
CURRENT_STAGE='dominant_client'
def _dominant_client():
    global clients, trust_threshold
    rows=[]; reason='disabled or MAX_CLIENT_SHARE >= 1'
    if not (CONFIG.get('RUN_DOMINANT_CLIENT_SCENARIO',True) and CONFIG.get('MAX_CLIENT_SHARE',1.0)<1): return rows,reason
    _saved={'clients':clients,'trust_threshold':trust_threshold}; _saved_gate=dict(GATE_STATE)
    try:
        clients=dirichlet_partition(train_idx,y,CONFIG['N_CLIENTS'],CONFIG['DIRICHLET_ALPHA'],CONFIG['SEED'],max_share=1.0)
        shares=np.array([len(c) for c in clients],dtype=float); shares=shares/shares.sum()
        print('uncapped client shares:',np.round(shares,3).tolist())
        if shares.max()<=CONFIG['MAX_CLIENT_SHARE']+1e-9:
            return [],'cap never binds on this split: uncapped == capped, scenario not needed'
        _dc=calibrate_trust(CONFIG['TRUST_UPDATE'],CONFIG['SEED']+2777)
        trust_threshold=_dc['tau']; GATE_STATE['median_cal']=_dc['median']
        for mi,m in enumerate(methods):
            mdl,_,tl,secs,comm=run_federated(m,CONFIG['SEED']+mi*100)
            r=metric_row(mdl,test_idx,m,secs,comm)
            r.update({'Scenario':'uncapped Dirichlet split (dominant client)','LargestClient':int(shares.argmax()),
                      'LargestClientShare':float(shares.max()),'Tau_cal_scenario':_dc['tau']})
            if len(tl) and 'Accepted' in tl: r.update(admission_stats(tl))
            rows.append(r)
        return rows,''
    finally:
        clients=_saved['clients']; trust_threshold=_saved['trust_threshold']; GATE_STATE.clear(); GATE_STATE.update(_saved_gate)

dom_rows,_dom_reason=run_stage('dominant_client',_dominant_client)
if not dom_rows: print('Dominant-client scenario not run:',_dom_reason)
dom_df=pd.DataFrame(dom_rows) if dom_rows else pd.DataFrame([{'Status':'NOT_RUN','Reason':_dom_reason}])
save_table(dom_df,'Table_13b_Dominant_Client_Scenario')
if dom_rows:
    _p=pd.DataFrame(dom_rows).merge(perf[['Method','MacroF1']].rename(columns={'MacroF1':'CappedSplit'}),on='Method')
    ax=_p.set_index('Method')[['CappedSplit','MacroF1']].rename(columns={'MacroF1':'UncappedDominantClient'}).plot(kind='bar',figsize=(9,4.5))
    ax.set_ylim(0,1.02); ax.set_ylabel('Test Macro-F1'); plt.xticks(rotation=25,ha='right'); plt.title('Capped split vs dominant-client split')
    savefig('Figure_10b_Dominant_Client_Scenario')


In [ ]:
# Cell 14 — Per-client fairness and trust-trajectory audit
client_rows=[]
for m,model in MODELS.items():
    for k,ids in enumerate(test_clients):
        if len(ids)<5: continue
        pr=np_predict(model,ids); yy=y[ids]; pred=pr.argmax(1); client_rows.append({'Method':m,'Client':k,'Rows':len(ids),'MacroF1':f1_score(yy,pred,average='macro',zero_division=0),'Accuracy':accuracy_score(yy,pred)})
client_perf=pd.DataFrame(client_rows); save_table(client_perf,'Table_14_Per_Client_Fairness')
fair=[]
for m,g in client_perf.groupby('Method'): fair.append({'Method':m,'MeanClientF1':g.MacroF1.mean(),'WorstClientF1':g.MacroF1.min(),'ClientF1_SD':g.MacroF1.std(ddof=1) if len(g)>1 else 0.0,'Range':g.MacroF1.max()-g.MacroF1.min()})
save_table(pd.DataFrame(fair),'Table_15_Fairness_Summary')
plt.figure(figsize=(8,4.5))
for m,g in client_perf.groupby('Method'): plt.plot(g.Client,g.MacroF1,marker='o',label=m)
plt.xlabel('Client'); plt.ylabel('Macro-F1'); plt.ylim(0,1.02); plt.legend(fontsize=7,ncol=2); plt.title('Client-level fairness under non-IID data'); savefig('Figure_11_Client_Fairness')
if len(trustlog):
    g=trustlog[trustlog.Method=='AgriTrust-ACFL']
    plt.figure(figsize=(8,4.5))
    for c,h in g.groupby('Client'): plt.plot(h.Round,h.UpdatedTrust,marker='o',label=f'C{c}')
    plt.axhline(trust_threshold,linestyle='--',color='k',label='tau_cal'); eff=g.groupby('Round').EffectiveThreshold.first(); plt.step(eff.index,eff.values,where='mid',color='grey',label='effective gate'); plt.xlabel('Round'); plt.ylabel('Trust'); plt.ylim(0,1.05); plt.legend(fontsize=7,ncol=2); plt.title('Client trust trajectory during clean federation'); savefig('Figure_12_Clean_Trust_Trajectories')


In [ ]:
# Cell 15 — Cumulative trust-component ablation through the real pipeline
CURRENT_STAGE='ablation'
# All rows start from AgriTrust's own warm-up model with FRESH trust, so components are compared on equal footing.
ABLATION_LADDER=[
    ('A0 Staleness-weighted only',frozenset(),None),
    ('A1 +Direction',frozenset({'direction'}),None),
    ('A2 +Norm',frozenset({'direction','norm'}),None),
    ('A3 +Reliability+History',frozenset({'direction','norm','reliability','history'}),None),
    ('A4 +Clipping',frozenset({'direction','norm','reliability','history','clip'}),None),
    ('A5 +Similarity (duplicate-aware consensus)',frozenset({'direction','norm','reliability','history','clip','similarity'}),None),
    ('A6 +Gate (no probation)',frozenset({'direction','norm','reliability','history','clip','similarity','gate'}),None),
    (f"A7 +Probation (full AgriTrust, {CONFIG['TRUST_UPDATE']} memory)",TRUST_COMPONENTS_FULL,None),
]
#the non-selected trust-memory rule as its own row, run with ITS OWN calibrated tau and median
for _alt in [m for m in TRUST_MODES if m!=CONFIG['TRUST_UPDATE']]:
    ABLATION_LADDER.append((f'A8 Full AgriTrust, {_alt} memory',TRUST_COMPONENTS_FULL,_alt))
def _ablation():
    abl=[]
    for ks in range(CONFIG.get('ABLATION_SEEDS_N',1)):   #several seeds in FULL_REAL
      for si,(at,ratio) in enumerate(CONFIG['ABLATION_SCENARIOS']):
        sd=8800+si+1000*ks
        bad=pick_bad(ratio,sd) if at!='none' else set()
        attack=None if at=='none' else {'type':at,'ratio':ratio,'bad':bad}
        for name,comp,tmode in ABLATION_LADDER:
            mdl,_,tl,secs,_=run_federated('AgriTrust-ACFL',sd,rounds=CONFIG['ABLATION_ROUNDS'],init_state=WARM_STATE,attack=attack,components=comp,
                                          trust_update=tmode,threshold=(TRUST_CAL[tmode]['tau'] if tmode else None),
                                          median_cal=(TRUST_CAL[tmode]['median'] if tmode else None))
            rr=metric_row(mdl,test_idx,name,secs); a=admission_stats(tl) if 'gate' in comp else {}
            rr['MaxConsecutiveBenignQuarantine']=a.get('MaxConsecutiveBenignQuarantine',0); rr['ProbationAdmitsMalicious']=a.get('ProbationAdmitsMalicious',0)
            rr.update({'Seed':sd,'Scenario':'clean' if at=='none' else f'{at}@{ratio}',
                       'MeanTrustMalicious':tl[tl.Malicious].UpdatedTrust.mean() if len(bad) else np.nan,
                       'MeanTrustBenign':tl[~tl.Malicious].UpdatedTrust.mean(),
                       'BenignFalseQuarantine':a.get('BenignFalseQuarantine',0.0),
                       'MaliciousQuarantineRecall':a.get('MaliciousQuarantineRecall',np.nan)})
            abl.append(rr)
    return abl
abl=run_stage('ablation',_ablation)
ablation=pd.DataFrame(abl); save_table(ablation,'Table_16_Trust_Component_Ablation')
save_table(ablation.groupby(['Method','Scenario'],sort=False).MacroF1.agg(['mean','std','count']).reset_index(),'Table_16c_Ablation_Mean_SD_over_Seeds')
piv=ablation.pivot_table(index='Method',columns='Scenario',values='MacroF1',aggfunc='mean').reindex([x[0] for x in ABLATION_LADDER])
ax=piv.plot(kind='bar',figsize=(9,4.5)); ax.set_ylim(0,1.02); ax.set_ylabel('Test Macro-F1'); plt.xticks(rotation=25,ha='right'); plt.legend(fontsize=7)
plt.title('Cumulative trust-component ablation'); savefig('Figure_13_Trust_Ablation')


## Validation-only hyperparameter sensitivity
This section audits τ (trust admission threshold) and λ (staleness decay) on the validation partition only. It is not post-hoc test tuning.


In [ ]:
CURRENT_STAGE='sensitivity'
sensitivity_rows=[]
def _sens_run(param,value,seed,**kw):
    for si,(at,ratio) in enumerate(CONFIG['SENS_SCENARIOS']):
        bad=pick_bad(ratio,9100+si) if at!='none' else set()
        attack=None if at=='none' else {'type':at,'ratio':ratio,'bad':bad}
        mdl,_,tl,_,_=run_federated('AgriTrust-ACFL',seed+si,rounds=CONFIG['SENS_ROUNDS'],init_state=WARM_STATE,init_trust=WARM_TRUST,attack=attack,**kw)
        r=metric_row(mdl,val_idx,'AgriTrust-ACFL'); a=admission_stats(tl)
        sensitivity_rows.append({'Parameter':param,'Value':float(value),'Scenario':'clean' if at=='none' else f'{at}@{ratio}',
                                 'ValMacroF1':r['MacroF1'],'ValMacroAUROC':r['MacroAUROC'],'ValMacroFPR':r['MacroFPR'],**a})

tau_grid=sorted({round(float(t),3) for t in CONFIG['TAU_SENSITIVITY']}|{round(float(trust_threshold),3)})   # always includes tau_cal
print('tau grid:',tau_grid,'| tau_cal =',round(trust_threshold,3))
def _sensitivity():
    sensitivity_rows.clear()
    for tau in tau_grid: _sens_run('tau_absolute',tau,9123,threshold=float(tau),gate_mode='absolute')
    for tau in tau_grid: _sens_run('tau_hybrid',tau,9123,threshold=float(tau),gate_mode='hybrid')
    orig_lambda=CONFIG['STALENESS_LAMBDA']
    try:
        for lam in CONFIG['LAMBDA_SENSITIVITY']:
            CONFIG['STALENESS_LAMBDA']=float(lam); _sens_run('lambda',lam,9123)
    finally:
        CONFIG['STALENESS_LAMBDA']=orig_lambda
    return list(sensitivity_rows)
sensitivity_rows[:]=run_stage('sensitivity',_sensitivity)

sensitivity_df=pd.DataFrame(sensitivity_rows)
save_table(sensitivity_df,'Table_16b_Tau_Lambda_Sensitivity_Validation')
for param in sensitivity_df.Parameter.unique():
    g=sensitivity_df[sensitivity_df.Parameter==param]
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    for sc,h in g.groupby('Scenario'):
        axs[0].plot(h.Value,h.ValMacroF1,marker='o',label=sc); axs[1].plot(h.Value,h.BenignFalseQuarantine,marker='o',label=sc)
    axs[0].set_ylabel('Validation Macro-F1'); axs[1].set_ylabel('Benign false-quarantine rate')
    for ax in axs: ax.set_xlabel(param); ax.set_ylim(0,1.02); ax.legend(fontsize=7)
    plt.suptitle(f'Validation sensitivity: {param}'); savefig(f'Figure_13b_Sensitivity_{param}')


In [ ]:
CURRENT_STAGE='multiseed'
def seed_statistics(seed_df,scenario_label):
    """Per-method mean/SD/95% CI, Friedman omnibus, Wilcoxon AgriTrust vs each baseline with Holm correction and dz effect size."""
    desc=[]; stat_rows=[]
    melted=seed_df.melt(id_vars=['Method','Seed'],value_vars=['MacroF1','MacroAUROC','MacroFPR'],var_name='Metric',value_name='Value')
    for (m,metric),g in melted.groupby(['Method','Metric']):
        x=g.Value.dropna().values
        ci=stats.t.interval(.95,len(x)-1,loc=np.mean(x),scale=stats.sem(x)) if len(x)>1 else (np.nan,np.nan)
        desc.append({'Scenario':scenario_label,'Method':m,'Metric':metric,'NSeeds':len(x),'Mean':np.mean(x),
                     'SD':np.std(x,ddof=1) if len(x)>1 else 0.0,'CI95_Low':ci[0],'CI95_High':ci[1]})
    for metric in ['MacroF1','MacroAUROC','MacroFPR']:
        p=seed_df.pivot(index='Seed',columns='Method',values=metric).dropna()
        fr_p=np.nan
        if p.shape[0]>=3 and p.shape[1]>=3:
            fr=stats.friedmanchisquare(*[p[c].values for c in p.columns]); fr_p=fr.pvalue
            stat_rows.append({'Scenario':scenario_label,'Metric':metric,'Test':'Friedman','Comparison':'All methods',
                              'Statistic':fr.statistic,'p_value':fr.pvalue,'EffectSize':np.nan,'NSeeds':len(p)})
        if 'AgriTrust-ACFL' in p.columns:
            for m in [x for x in p.columns if x!='AgriTrust-ACFL']:
                d=p['AgriTrust-ACFL']-p[m]
                if len(d)>=5:
                    try:
                        w=stats.wilcoxon(p['AgriTrust-ACFL'],p[m],alternative='two-sided',zero_method='wilcox')
                        stat_rows.append({'Scenario':scenario_label,'Metric':metric,'Test':'Wilcoxon','Comparison':f'AgriTrust-ACFL vs {m}',
                                          'Statistic':w.statistic,'p_value':w.pvalue,'EffectSize':float(d.mean()/(d.std(ddof=1)+1e-12)),
                                          'MeanDiff':float(d.mean()),'NSeeds':len(d),'OmnibusFriedman_p':fr_p})
                    except Exception: pass
    st=pd.DataFrame(stat_rows)
    if len(st):
        st['p_Holm']=st['p_value']
        for metric,g in st[st.Test=='Wilcoxon'].groupby('Metric'):
            ix=g.index.tolist(); ps=st.loc[ix,'p_value'].values.astype(float); order=np.argsort(ps); adj=np.empty_like(ps); running=0.0
            for rank,pos in enumerate(order):
                running=max(running,min(1.0,(len(ps)-rank)*ps[pos])); adj[pos]=running
            st.loc[ix,'p_Holm']=adj
    return pd.DataFrame(desc),st

seed_rows=[]; poison_seed_rows=[]
PROG=CKPT/'multiseed'; PROG.mkdir(parents=True,exist_ok=True); print('Seed progress folder:',PROG)
if CONFIG['RUN_MULTI_SEED']:
    for sd in CONFIG['PAPER_SEEDS']:
        f=PROG/f'clean_seed_{sd}.csv'
        if f.exists(): seed_rows.append(pd.read_csv(f)); print('resume: clean seed',sd,'loaded'); continue
        rows=[]
        for i,m in enumerate(methods):
            mdl,_,_,secs,comm=run_federated(m,sd+i*1000); r=metric_row(mdl,test_idx,m,secs,comm); r['Seed']=sd; rows.append(r)
        d=pd.DataFrame(rows); d.to_csv(f,index=False); seed_rows.append(d); print('clean seed',sd,'done')
    seed_df=pd.concat(seed_rows,ignore_index=True); save_table(seed_df,'Table_17_MultiSeed_Robustness')
    # Seed-level poisoning replication: new warm start, attacker set and selection stream per seed.
    for sd in CONFIG['PAPER_SEEDS'][:CONFIG['POISON_SEEDS_N']]:
        f=PROG/f'poison_seed_{sd}.csv'
        if f.exists(): poison_seed_rows.append(pd.read_csv(f)); print('resume: poisoning seed',sd,'loaded'); continue
        ws=method_warm_starts(sd+4242,POISON_METHODS)
        parts=[]
        for at,ratio in CONFIG['MULTISEED_POISON_SCENARIOS']:
            pdf,_=poison_experiment(seed=sd,warm_states=ws,ratios=[ratio],types=[at],include_none=False)
            pdf['Seed']=sd; pdf['Scenario']=f'{at}@{ratio}'; parts.append(pdf)
        d=pd.concat(parts,ignore_index=True); d.to_csv(f,index=False); poison_seed_rows.append(d); print('poisoning seed',sd,'done')
    poison_seed_df=pd.concat(poison_seed_rows,ignore_index=True); save_table(poison_seed_df,'Table_17c_MultiSeed_Poisoning')
else:
    seed_df=pd.DataFrame(); poison_seed_df=pd.DataFrame()
    save_table(pd.DataFrame([{'Status':'NOT_RUN','Reason':'Set QUICK_RUN=False or RUN_MULTI_SEED=True for publication inference.'}]),'Table_17_MultiSeed_Robustness')
    save_table(pd.DataFrame([{'Status':'NOT_RUN','Reason':'Seed-level poisoning replication requires RUN_MULTI_SEED=True.'}]),'Table_17c_MultiSeed_Poisoning')

if len(seed_df):
    descs=[]; sts=[]
    d0,s0=seed_statistics(seed_df,'clean'); descs.append(d0); sts.append(s0)
    for sc,g in poison_seed_df.groupby('Scenario'):
        d1,s1=seed_statistics(g,sc); descs.append(d1); sts.append(s1)
    save_table(pd.concat(descs,ignore_index=True),'Table_17b_MultiSeed_Summary_CI')
    stats_df=pd.concat(sts,ignore_index=True); save_table(stats_df,'Table_18_Statistical_Tests')
else:
    stats_df=pd.DataFrame([{'Status':'NOT_RUN','Reason':'Independent multi-seed run required. Client rows are not inferential units.'}])
    save_table(stats_df,'Table_18_Statistical_Tests')


In [ ]:
# Cell 17 — Runtime/communication analysis and multi-criteria summary
comp=perf[['Method','TrainSeconds','CommMB','Parameters','MacroF1','MacroAUROC','MacroFPR']].copy(); save_table(comp,'Table_19_Computation_and_Communication')
plt.figure(figsize=(8,4)); plt.bar(comp.Method,comp.TrainSeconds); plt.xticks(rotation=25,ha='right'); plt.ylabel('Training seconds'); plt.title('Federated training runtime'); savefig('Figure_14_Training_Time')
plt.figure(figsize=(7,5)); plt.scatter(comp.CommMB,comp.MacroF1)
for _,r in comp.iterrows(): plt.text(r.CommMB,r.MacroF1,r.Method,fontsize=7)
plt.xlabel('Communication MB'); plt.ylabel('Macro-F1'); plt.title('Performance–communication trade-off'); savefig('Figure_15_Performance_vs_Communication')


## Independent external validation — UNSW-NB15
Separate external study. Because the feature schema differs from CIC-IoT-2023, preprocessing and fitting are repeated on official UNSW-NB15 train/test files; schemas are never pooled.


In [ ]:
external_rows=[]; UNSW_READY=False
if CONFIG.get('RUN_EXTERNAL_UNSW',False):
    trp=Path(CONFIG['UNSW_TRAIN']); tep=Path(CONFIG['UNSW_TEST'])
    if trp.exists() and tep.exists():
        utr=pd.read_csv(trp,low_memory=False); ute=pd.read_csv(tep,low_memory=False)
        # official partition sizes are 175,341 (training-set) and 82,332 (testing-set); mirrors sometimes differ
        UNSW_ROWS=(len(utr),len(ute)); UNSW_ROWS_OK=UNSW_ROWS==(175341,82332)
        print('UNSW-NB15 rows (train, test):',UNSW_ROWS,'| matches official sizes:',UNSW_ROWS_OK,'| source:',CONFIG.get('UNSW_SOURCE_NOTE'))
        if not UNSW_ROWS_OK: print('WARNING: UNSW-NB15 row counts differ from the official partition files - check the mirror before reporting.')
        target='attack_cat' if 'attack_cat' in utr.columns else ('label' if 'label' in utr.columns else None)
        if target is None:
            raise RuntimeError('UNSW-NB15 target column not found (expected attack_cat or label).')

        shared=[c for c in utr.columns if c in ute.columns and c!=target and pd.api.types.is_numeric_dtype(utr[c])]
        shared=[c for c in shared if c.lower() not in {'id','label'}]
        if len(shared)<4:
            raise RuntimeError('Too few compatible numeric UNSW-NB15 features.')

        Utr=utr[shared].replace([np.inf,-np.inf],np.nan).copy()
        Ute=ute[shared].replace([np.inf,-np.inf],np.nan).copy()
        med=Utr.median(numeric_only=True); Utr=Utr.fillna(med); Ute=Ute.fillna(med)
        scaler_u=StandardScaler().fit(Utr.values)
        Xutr=scaler_u.transform(Utr.values).astype('float32')
        Xute=scaler_u.transform(Ute.values).astype('float32')

        UNSW_READY=True; unsw_feats=list(shared)
        if target=='label':
            ytr_u=utr[target].astype(int).values
            yte_u=ute[target].astype(int).values
            uclasses=['Benign','Attack']
        else:
            lab_tr=utr[target].fillna('Normal').astype(str)
            lab_te=ute[target].fillna('Normal').astype(str)
            uclasses=sorted(lab_tr.unique()); umap={c:i for i,c in enumerate(uclasses)}
            keep=lab_te.isin(umap)
            Xute=Xute[keep.values]; lab_te=lab_te[keep]
            ytr_u=np.array([umap[x] for x in lab_tr],dtype=np.int64)
            yte_u=np.array([umap[x] for x in lab_te],dtype=np.int64)

        class UNSWMLP(nn.Module):
            def __init__(self,p,k,h=64):
                super().__init__()
                self.net=nn.Sequential(nn.Linear(p,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(.1),
                                       nn.Linear(h,h),nn.GELU(),nn.Linear(h,k))
            def forward(self,x): return self.net(x)

        seed_all(CONFIG['SEED']+900)
        um=UNSWMLP(Xutr.shape[1],len(np.unique(ytr_u)),CONFIG['D_MODEL']).to(DEVICE)
        opt=torch.optim.AdamW(um.parameters(),lr=1e-3,weight_decay=1e-4)
        ds=TensorDataset(torch.tensor(Xutr),torch.tensor(ytr_u))
        dl=DataLoader(ds,batch_size=512,shuffle=True)
        epochs=2 if CONFIG['QUICK_RUN'] else 8
        um.train()
        for ep in range(epochs):
            for xb,yb in dl:
                xb=xb.to(DEVICE); yb=yb.to(DEVICE)
                opt.zero_grad(); loss=nn.CrossEntropyLoss()(um(xb),yb); loss.backward(); opt.step()

        um.eval(); pp=[]
        with torch.no_grad():
            for s in range(0,len(Xute),4096):
                pp.append(torch.softmax(um(torch.tensor(Xute[s:s+4096],device=DEVICE)),1).cpu().numpy())
        up=np.vstack(pp); uy=up.argmax(1)
        try:
            uauc=(roc_auc_score(yte_u,up,multi_class='ovr',average='macro')
                  if up.shape[1]>2 else roc_auc_score(yte_u,up[:,1]))
        except Exception:
            uauc=np.nan
        external_rows.append({
            'Dataset':'UNSW-NB15','Method':'Centralized MLP reference (not federated)','Scenario':'clean','TrainRows':len(Xutr),'TestRows':len(Xute),'Features':Xutr.shape[1],
            'Classes':len(np.unique(ytr_u)),'Accuracy':accuracy_score(yte_u,uy),
            'BalancedAccuracy':balanced_accuracy_score(yte_u,uy),
            'MacroF1':f1_score(yte_u,uy,average='macro',zero_division=0),'MacroAUROC':uauc,
            'Protocol':'Independent external dataset; preprocessing/model fitting repeated on UNSW train'
        })
    else:
        external_rows.append({'Dataset':'UNSW-NB15','Status':'NOT_RUN',
                              'Reason':'Official UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv not found. Stage them in Step 0c ('+CONFIG['UNSW_TRAIN']+').'})
else:
    external_rows.append({'Dataset':'UNSW-NB15','Status':'DISABLED',
                          'Reason':'Set RUN_EXTERNAL_UNSW=True for external validation.'})



In [ ]:
# External validation (b) — the SAME federated protocol replayed on UNSW-NB15 (single seed; descriptive)
CURRENT_STAGE='unsw_federated'
# Globals used by run_federated are temporarily re-bound to UNSW-NB15 and restored afterwards.
# Trust is re-calibrated on a benign UNSW warm-up (the CIC-IoT-2023 tau is never reused).
UNSW_METHODS=['FedAvg','FedProx','Async-Stale','CoordMedian','TrimmedMean','MultiKrum','RawTrust','AgriTrust-ACFL']
def _unsw_federated():
    global X, y, num, classes, clients, train_idx, val_idx, test_idx, criterion, trust_threshold
    rows=[]
    _keys=['X','y','num','classes','clients','train_idx','val_idx','test_idx','criterion','trust_threshold']
    _saved={k:globals()[k] for k in _keys}; _saved_gate=dict(GATE_STATE)
    try:
        ntr=len(Xutr); X=np.vstack([Xutr,Xute]).astype('float32'); y=np.concatenate([ytr_u,yte_u]).astype(np.int64)
        num=list(unsw_feats); classes=[str(c) for c in uclasses]
        tr_all=np.arange(ntr); test_idx=np.arange(ntr,ntr+len(Xute))
        try: train_idx,val_idx=train_test_split(tr_all,test_size=CONFIG['VAL_FRAC_FROM_TRAIN'],stratify=ytr_u,random_state=CONFIG['SEED']+517)
        except ValueError: train_idx,val_idx=train_test_split(tr_all,test_size=CONFIG['VAL_FRAC_FROM_TRAIN'],random_state=CONFIG['SEED']+517)
        clients=dirichlet_partition(train_idx,y,CONFIG['N_CLIENTS'],CONFIG['DIRICHLET_ALPHA'],CONFIG['SEED']+500)
        _cnt=np.bincount(y[train_idx],minlength=len(classes)).astype(float)
        criterion=nn.CrossEntropyLoss(weight=torch.tensor(_cnt.sum()/(len(classes)*np.maximum(_cnt,1)),dtype=torch.float32,device=DEVICE))
        _uc=calibrate_trust(CONFIG['TRUST_UPDATE'],CONFIG['SEED']+1777)     # same routine (burn-in, selected rule)
        trust_threshold=_uc['tau']; GATE_STATE['median_cal']=_uc['median']; unsw_tau=trust_threshold
        uws_all=method_warm_starts(CONFIG['SEED']+4343,[m for m in UNSW_METHODS if m!='FedProx'])   # per-method
        for si,(at,ratio) in enumerate([('none',0.0)]+list(CONFIG['UNSW_POISON_SCENARIOS'])):
            bad=pick_bad(ratio,CONFIG['SEED']+610+si) if at!='none' else set()
            for meth in UNSW_METHODS:
                if at=='none':
                    mdl,_,tl,secs,comm=run_federated(meth,CONFIG['SEED']+600,rounds=CONFIG['UNSW_FED_ROUNDS'])
                else:
                    if meth=='FedProx': continue
                    mdl,_,tl,secs,comm=run_federated(meth,CONFIG['SEED']+600+si,rounds=CONFIG['POISON_ROUNDS'],init_state=uws_all[meth]['state'],init_trust=uws_all[meth]['trust'],
                                                     attack={'type':at,'ratio':ratio,'bad':bad})
                r=metric_row(mdl,test_idx,meth,secs,comm)
                r.update({'Dataset':'UNSW-NB15','Scenario':'clean' if at=='none' else f'{at}@{ratio}','Tau_cal_UNSW':unsw_tau,
                          'UNSWSource':CONFIG.get('UNSW_SOURCE_NOTE'),'UNSWRowsMatchOfficial':bool(globals().get('UNSW_ROWS_OK',False)),
                          'Protocol':'Same federated protocol on UNSW-NB15; non-IID Dirichlet clients; trust re-calibrated on UNSW warm-up; single seed'})
                if meth=='AgriTrust-ACFL' and len(tl): r.update(admission_stats(tl))
                rows.append(r)
        print('UNSW-NB15 federated replication done; tau_cal(UNSW) =',round(unsw_tau,4))
        return rows
    finally:
        globals().update(_saved); GATE_STATE.clear(); GATE_STATE.update(_saved_gate)

if CONFIG.get('RUN_EXTERNAL_UNSW',False) and UNSW_READY:
    external_rows.extend(run_stage('unsw_federated',_unsw_federated))

external_df=pd.DataFrame(external_rows)
save_table(external_df,'Table_21_External_UNSWNB15_Validation')
if 'MacroF1' in external_df.columns and external_df['MacroF1'].notna().any():
    g=external_df[external_df['MacroF1'].notna() & external_df.get('Method',pd.Series(dtype=str)).isin(UNSW_METHODS)]
    if len(g):
        piv=g.pivot_table(index='Method',columns='Scenario',values='MacroF1').reindex([m for m in UNSW_METHODS if m in set(g.Method)])
        ax=piv.plot(kind='bar',figsize=(9,4.5)); ax.set_ylim(0,1.02); ax.set_ylabel('UNSW-NB15 test Macro-F1'); plt.xticks(rotation=25,ha='right'); plt.legend(fontsize=7)
        plt.title('External replication on UNSW-NB15 (single seed)'); savefig('Figure_16_External_UNSWNB15_Validation')


In [ ]:

# Claim eligibility / integrity gate
real_data = not provenance.startswith('SYNTHETIC_SMOKE')
checks=[]
def check(name,ok,evidence,boundary=''):
    checks.append({'Claim_or_Check':name,'Eligible':bool(ok),'Evidence':evidence,'Boundary':boundary})

check('Real verified primary benchmark used',real_data,'Table_00a, Table_00',
      'Synthetic smoke is never manuscript evidence.')
check('Leakage-safe train/validation/calibration/test separation',
      real_data and len(train_idx)>0 and len(val_idx)>0 and len(cal_idx)>0 and len(test_idx)>0,
      'Table_00 / preprocessing logs','Preprocessing must be fitted on training rows only.')
check('Non-IID 10-client federation',real_data and CONFIG['N_CLIENTS']>=10,
      'Table_04_NonIID_Client_Distribution.csv')
check('Staleness/dropout robustness',real_data and len(conn_df)>0,
      'Table_13_Connectivity_Robustness.csv',
      'Controlled connectivity stress, not natural field telemetry.')
check('Multiple Byzantine attack types/ratios',
      real_data and len(CONFIG['POISON_TYPES'])>=4 and len(CONFIG['POISON_RATIOS'])>=3,
      'Tables_09-11')
check('Adaptive adversary evaluated',
      real_data and 'adaptive_slow_drift' in CONFIG['POISON_TYPES'],
      'Tables_09-11',
      'Adaptive slow-drift is a controlled proxy, not an exhaustive adaptive threat model.')
check('Benign false-quarantine audit',real_data and len(noise_audit)>0,
      'Table_12_Benign_Noise_Staleness_Trust_Audit.csv')
check('Tau/lambda sensitivity analysis',
      real_data and 'sensitivity_df' in globals() and len(sensitivity_df)>0,
      'Table_16b + Figure_13b')
check('Trust-component ablation',real_data and len(ablation)>0,
      'Table_16_Trust_Component_Ablation.csv')
check('Independent multi-seed clean inference (>=10 seeds; Holm-corrected Wilcoxon needs >=9 for 7 comparisons)',
      real_data and CONFIG['RUN_MULTI_SEED'] and len(seed_df)>0 and seed_df.Seed.nunique()>=10,
      'Tables_17,17b,18 (Scenario=clean)',
      'Client rows are not inferential units.')
check('Independent multi-seed POISONING inference (>=10 seeds; Holm-corrected Wilcoxon)',
      real_data and CONFIG['RUN_MULTI_SEED'] and len(poison_seed_df)>0 and poison_seed_df.Seed.nunique()>=10,
      'Tables_17c,17b,18 (poisoning scenarios)',
      'Headline robustness claims require this row to be Eligible.')
_fq=float(clean_admission.loc[clean_admission.Method=='AgriTrust-ACFL','BenignFalseQuarantine'].iloc[0]) if len(clean_admission) and (clean_admission.Method=='AgriTrust-ACFL').any() else np.nan
check(f'Clean-run benign false quarantine reported (AgriTrust = {_fq:.3f})',real_data and not np.isnan(_fq),
      'Table_08b_Clean_Run_Admission_Audit',
      'Report this value alongside every robustness claim; do not claim low false quarantine if it is high.')
check('Primary data from official/local files (not a convenience mirror)',
      real_data and 'Kaggle' not in provenance,'Table_00a, Table_00b',
      'If a mirror was used, state it and verify file hashes/labels against the official release.')
check('Exact duplicates removed before splitting; cross-split identical-feature rows audited',
      real_data and CONFIG.get('DROP_EXACT_DUPLICATES',False),'Table_00 (ExactDuplicatesRemoved, TestRowsWithIdenticalTrainFeatures)',
      'Report both counts in the leakage-control section.')
check('Independent external UNSW-NB15 federated replication (single seed; descriptive only)',
      real_data and CONFIG.get('RUN_EXTERNAL_UNSW',False) and 'external_df' in globals()
      and 'Method' in external_df.columns and (external_df.get('Method')=='AgriTrust-ACFL').any(),
      'Table_21, Figure_16',
      'Separate preprocessing and re-calibrated trust; features never pooled with CIC-IoT-2023; no inferential claims from one seed.')
check('Communication/runtime evidence',real_data and len(comp)>0,'Table_19')
check('Formal differential-privacy guarantee',False,'Not implemented',
      'Do not infer ε,δ privacy from Gaussian update-noise experiments.')
check('Physical Raspberry Pi / ESP32 / farm-gateway validation',False,'Not implemented',
      'Do not claim physical-edge deployment without measured hardware evidence.')
check('Secure aggregation compatibility',False,'Not implemented',
      'Per-client trust scoring observes individual updates; this conflicts with standard secure aggregation.')
check('True event-driven asynchronous FL',False,'Not implemented',
      'Describe implementation as staleness-aware buffered federation unless a true event queue is added.')

claim=pd.DataFrame(checks)
save_table(claim,'Table_20_Claim_Eligibility_and_Integrity_Gate')
print(claim.to_string(index=False))

# diagnostic acceptance: must PASS on the QUICK_REAL run before the expensive FULL_REAL run
diag=[]
def dx(name,ok,detail): diag.append({'Diagnostic':name,'Status':'PASS' if ok else 'CHECK','Detail':detail})
_pf=perf.set_index('Method').MacroF1
dx('AgriTrust clean Macro-F1 within 0.03 of Async-Stale (same buffered protocol)',
   _pf['AgriTrust-ACFL']>=_pf['Async-Stale']-0.03,f"AgriTrust={_pf['AgriTrust-ACFL']:.4f}, Async-Stale={_pf['Async-Stale']:.4f}, best={_pf.idxmax()} {_pf.max():.4f}")
dx('AgriTrust clean-run benign false quarantine <= 10%',not np.isnan(_fq) and _fq<=0.10,f'{_fq:.3f} (Table_08b)')
_ta=sensitivity_df[(sensitivity_df.Parameter=='tau_absolute')&(sensitivity_df.Scenario=='clean')]
dx('tau sweep actually binds (admitted fraction varies across tau under the absolute gate)',
   len(_ta)>1 and _ta.AdmittedFraction.nunique()>1,f"admitted fractions: {_ta.AdmittedFraction.round(3).tolist()}")
_c=conn_df.pivot_table(index=['DropoutRate','MaxStaleness'],columns='Method',values='MacroF1')
_gap=float((_c['Async-Stale']-_c['AgriTrust-ACFL']).max()) if {'Async-Stale','AgriTrust-ACFL'}<=set(_c.columns) else np.nan
dx('Connectivity: AgriTrust never more than 0.05 below Async-Stale in any dropout x staleness cell',
   not np.isnan(_gap) and _gap<=0.05,f'largest Async-Stale minus AgriTrust gap = {_gap:.4f} (Table_13)')
_reg=pd.DataFrame(RUN_REGISTRY); _a=_reg[(_reg.Method=='AgriTrust-ACFL')&_reg.Stage.isin(['main','connectivity','poisoning','multiseed'])]
_full='+'.join(sorted(TRUST_COMPONENTS_FULL))
dx('Main/connectivity/poisoning/multi-seed AgriTrust runs share one configuration (gate, components, lambda, tau, trust rule)',
   len(_a)>0 and _a.GateMode.eq(CONFIG['GATE_MODE']).all() and _a.Components.eq(_full).all()
   and _a.Lambda.nunique()==1 and _a.Threshold.nunique()==1 and _a.TrustUpdate.nunique()==1,
   f"gate={sorted(_a.GateMode.unique())}, lambda={sorted(_a.Lambda.unique())}, tau={sorted(_a.Threshold.unique())}, rule={sorted(_a.TrustUpdate.unique())}")
dx('Calibration rests on >= 50 warm-up client-rounds after burn-in',TRUST_CAL[CONFIG['TRUST_UPDATE']]['n']>=50,
   f"{TRUST_CAL[CONFIG['TRUST_UPDATE']]['n']} client-rounds ({CONFIG['TRUST_WARMUP_ROUNDS']} rounds, {TRUST_CAL[CONFIG['TRUST_UPDATE']]['burnin']} burn-in excluded)")
_mcq=clean_admission.set_index('Method').MaxConsecutiveBenignQuarantine.get('AgriTrust-ACFL',np.nan) if 'MaxConsecutiveBenignQuarantine' in clean_admission else np.nan
dx(f"No honest client locked out (benign quarantine streak <= {CONFIG['PROBATION_AFTER']} participations, clean run)",
   not pd.isna(_mcq) and _mcq<=CONFIG['PROBATION_AFTER'],f'longest benign quarantine streak = {_mcq} participations (Table_08b)')
_lg=CONFIG['LAMBDA_SENSITIVITY']; _edges={max(_lg)}|({min(_lg)} if min(_lg)>0 else set())   # 0 = natural floor, not a grid edge
dx('lambda not selected at an arbitrary edge of its grid (lambda = 0, no decay, is a natural floor)',
   CONFIG['STALENESS_LAMBDA'] not in _edges or not CONFIG.get('SELECT_LAMBDA_ON_VALIDATION',True),
   f"selected {CONFIG['STALENESS_LAMBDA']} from {CONFIG['LAMBDA_SENSITIVITY']} (edge = optimum may lie outside the grid; report it)")
_mx=poison_df[poison_df.AttackType!='none'].PoisonRatio.max()
_sd=poison_df[(poison_df.AttackType=='adaptive_slow_drift')&(poison_df.PoisonRatio==_mx)]
_dmg=float(_sd.AttackDamage.mean()) if 'AttackDamage' in _sd and len(_sd) else np.nan
_sdu=_sd[_sd.Method.isin(['FedAvg','Async-Stale'])]
_dmg=float(_sdu.AttackDamage.mean()) if 'AttackDamage' in _sdu and len(_sdu) else np.nan
# Measured on the UNDEFENDED methods: robust aggregators can show negative 'damage' because without attackers
# Multi-Krum / median still discard honest clients every round, while with attackers they discard the attackers.
dx('Slow-drift (ALIE) adversary is effective: damage to undefended FedAvg/Async-Stale >= 0.02 Macro-F1 at the highest ratio',
   not np.isnan(_dmg) and _dmg>=0.02,f"mean paired damage vs own no-attack continuation = {_dmg:.4f} (Table_09 AttackDamage)")
_tdA=trust_detection[(trust_detection.Method=='AgriTrust-ACFL')&(trust_detection.AttackType=='adaptive_slow_drift')]
_fqA=float(_tdA.BenignFalseQuarantine.max()) if len(_tdA) else np.nan
dx('Colluding ALIE copies do not turn the gate against honest clients (benign false quarantine <= 15% at every ratio)',
   not np.isnan(_fqA) and _fqA<=0.15,f'max benign false quarantine under ALIE = {_fqA:.3f} ( 0.739 at 40%)')
_cl=TRUST_CAL[CONFIG['TRUST_UPDATE']]['log']
_ms=float(_cl.loc[_cl.UsedForCalibration,'MaxSimilarity'].quantile(.99)) if 'MaxSimilarity' in _cl and len(_cl) else np.nan
dx('Honest updates stay below the similarity threshold (warm-up 99th percentile < SIM_START)',
   not np.isnan(_ms) and _ms<CONFIG['SIM_START'],f"honest max-cosine p99 = {_ms:.4f}; SIM_START = {CONFIG['SIM_START']}")
_w=pd.read_csv(TAB/'Table_09a_Per_Method_Warm_Start_Checkpoints.csv')
dx('Attacks start from non-degenerate checkpoints (>= 5 of 7 methods with warm validation Macro-F1 >= 0.40)',
   int((_w.WarmValMacroF1>=0.40).sum())>=5,'; '.join(f'{m}={v:.3f}' for m,v in zip(_w.Method,_w.WarmValMacroF1)))
_sh=resource_df.TrainShare.max() if 'TrainShare' in resource_df else np.nan
# cap is ceil(MAX_CLIENT_SHARE x N) ROWS, so the share can exceed the fraction by < 1 row (0.250 failed)
_capr=int(np.ceil(CONFIG['MAX_CLIENT_SHARE']*resource_df.TrainRows.sum()))
dx(f"No client holds more than {CONFIG['MAX_CLIENT_SHARE']:.0%} of the training rows (capped split)",
   int(resource_df.TrainRows.max())<=_capr,f'largest client = {int(resource_df.TrainRows.max())} rows (share {_sh:.4f}); cap = {_capr} rows (Table_03)')
diag_df=pd.DataFrame(diag); save_table(diag_df,'Table_08c_Diagnostic_Acceptance'); print(diag_df.to_string(index=False))
reg=pd.DataFrame(RUN_REGISTRY)
save_table(reg.groupby(['Stage','Method','Init','InitTrust','GateMode','Threshold','Lambda','Components','TrustUpdate'],dropna=False).size().reset_index(name='Runs'),
           'Table_05d_Run_Configuration_Registry')


In [ ]:
# Cell 19 — Publication-quality architecture diagram
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
plt.figure(figsize=(12,6)); ax=plt.gca(); ax.axis('off')
boxes=[(0.02,.62,.14,.20,'Farm clients\nnon-IID traffic'),(0.20,.62,.14,.20,'Local IDS\ntraining'),(0.38,.62,.14,.20,'Async update\n+ staleness'),(0.56,.62,.14,.20,'Trust engine\ndirection • norm\nhistory • reliability'),(0.76,.62,.18,.20,'Clip • quarantine\ntrust-weighted\naggregation'),(0.30,.18,.18,.20,'Benign-noise /\nfalse-quarantine audit'),(0.54,.18,.18,.20,'Poisoning / collusion\nstress tests'),(0.78,.18,.18,.20,'Global IDS\n+ evidence gate')]
for x0,y0,w,h,t in boxes:
    ax.add_patch(FancyBboxPatch((x0,y0),w,h,boxstyle='round,pad=0.015',fill=False,linewidth=1.5)); ax.text(x0+w/2,y0+h/2,t,ha='center',va='center',fontsize=9)
def ar(a,b): ax.add_patch(FancyArrowPatch(a,b,arrowstyle='->',mutation_scale=14,linewidth=1.2))
for a,b in [((.16,.72),(.20,.72)),((.34,.72),(.38,.72)),((.52,.72),(.56,.72)),((.70,.72),(.76,.72)),((.65,.62),(.39,.38)),((.66,.62),(.63,.38)),((.85,.62),(.86,.38))]: ar(a,b)
ax.text(.5,.94,'AgriTrust-ACFL: calibrated client-trust federation for agricultural IoT security',ha='center',fontsize=14,fontweight='bold'); savefig('Figure_01_Proposed_Framework')


## Manuscript evidence pack — Abstract to References
Generates a structured map from manuscript section to executed evidence. It never invents numerical results.


In [ ]:

# Manuscript evidence pack — Abstract to References
sections = [
    ('Title','Frozen AgriTrust-ACFL title and method identity.'),
    ('Abstract','Use only executed headline results from Tables 06, 09-13, 17b, 18, 19, 21 and the claim gate.'),
    ('Keywords','Agricultural IoT; federated learning; intrusion detection; Byzantine robustness; client trust; non-IID; staleness.'),
    ('Highlights','Derive 4-5 evidence-backed points only after the final run.'),
    ('Introduction','Problem: benign heterogeneity/staleness can resemble malicious updates.'),
    ('Related Work','Compare FedTrans-AgriIDS, P4P, FedDBC, TrustFed-IDS, smart-city trust FL-IDS, FLEX-IDS.'),
    ('Research Gap','Use base-paper gap notes plus trust-calibration / false-quarantine positioning.'),
    ('Threat Model','Honest server; benign/stale/unavailable/resource-limited/Byzantine clients; secure-aggregation limitation.'),
    ('System Model','Farm gateways, local detector, buffered staleness-aware update ingestion, trust engine.'),
    ('Mathematical Formulation','Own-base update delta; direction/norm/reliability/history trust; EMA; freshness decay; own-delta clipping; drift-normalised admission gate tau_cal*min(1,median_t/median_cal) with quorum.'),
    ('Datasets','Tables 00a/00b (file manifest + hashes)/01b (label harmonization) + official CIC-IoT-2023 and UNSW-NB15 source pages.'),
    ('Leakage Control','Exact-duplicate removal before splitting; cross-split identical-feature audit (Table 00); training-only feature selection/imputation/scaling; integrity gate Table 00c.'),
    ('Non-IID Federation','Tables 03b/04 and client heatmap.'),
    ('Training / Validation / Testing','Table_00 split summary and configuration manifest.'),
    ('Baselines','FedAvg, FedProx, Async-Stale, CoordMedian, TrimmedMean, MultiKrum, RawTrust, AgriTrust.'),
    ('Main Results','Tables 06/06b-06f; Figures 03-06c.'),
    ('Trust Calibration','Tables 05/05b/08/08b; protocol-matched warm-up, hybrid gate, clean-run false quarantine.'),
    ('Poisoning Robustness','Tables 09-11; Figures 07-08.'),
    ('Adaptive Adversary','adaptive_slow_drift rows in Tables 09-11.'),
    ('False Quarantine','Table 12; Figure 09.'),
    ('Connectivity / Staleness','Table 13; Figure 10.'),
    ('Fairness','Tables 14-15; Figures 11-12.'),
    ('Sensitivity','Table 16b; Figure 13b_tau and Figure 13b_lambda.'),
    ('Ablation','Table 16; Figure 13.'),
    ('Statistics','Tables 17,17b,17c,18; independent seeds only; clean and poisoning scenarios.'),
    ('Computation / Communication','Table 19; Figures 14-15.'),
    ('External Validation','Table 21; Figure 16 — federated replication on UNSW-NB15 (single seed, descriptive).'),
    ('Discussion','State where AgriTrust helps, where it does not, and trade-offs.'),
    ('Limitations','No formal DP; no secure aggregation; no physical gateway measurements; buffered not true event-driven async.'),
    ('Conclusion','Only claims marked Eligible=True in Table 20.'),
    ('Data Availability','CIC-IoT-2023 UNB CIC official page; UNSW-NB15 official UNSW page.'),
    ('Code Availability','Notebook + output ZIP after execution.'),
    ('Ethics / Conflict / CRediT','Complete from authorship record; do not fabricate.'),
    ('References','Finalize with submission-time literature verification; include dataset papers and recent trust-FL comparators.')
]
evidence_pack=pd.DataFrame(sections,columns=['ManuscriptSection','EvidenceInstruction'])
save_table(evidence_pack,'Table_22_Manuscript_Abstract_to_References_Evidence_Map')

reference_checklist = [
    ['FedTrans-AgriIDS base paper','10.1007/s44163-026-01694-2'],
    ['CIC-IoT-2023 dataset','https://www.unb.ca/cic/datasets/iotdataset-2023.html'],
    ['UNSW-NB15 dataset','https://research.unsw.edu.au/projects/unsw-nb15-dataset'],
    ['P4P anti-poisoning FL-IDS','10.1016/j.jnca.2026.104502'],
    ['FedDBC collusion-resistant FIDS','10.1016/j.comnet.2026.112497'],
    ['TrustFed-IDS','10.1016/j.comnet.2026.112430'],
    ['Trust-aware smart-city FL-IDS','10.1016/j.comnet.2026.112616'],
    ['FLEX-IDS','10.1016/j.compeleceng.2025.110827'],
    ['ALIE attack: Baruch, Baruch & Goldberg, A Little Is Enough: Circumventing Defenses for Distributed Learning, NeurIPS 2019','verify DOI/proceedings link at submission'],
    ['FoolsGold: Fung, Yoon & Beschastnikh, The Limitations of Federated Learning in Sybil Settings, RAID 2020','verify DOI/proceedings link at submission']
]
save_table(pd.DataFrame(reference_checklist,columns=['ReferenceAnchor','DOI_or_OfficialURL']),
           'Table_23_Reference_Checklist')

md_lines=['# AgriTrust-ACFL manuscript evidence pack']
for sec,inst in sections:
    md_lines += ['',f'## {sec}',inst]
(OUT/'MANUSCRIPT_EVIDENCE_PACK.md').write_text('\n'.join(md_lines))


In [ ]:

# Final output index, manuscript-ready result summary and ZIP
index=[]
for p in sorted(TAB.glob('*.csv')):
    index.append({'Type':'Table','File':p.name,'Bytes':p.stat().st_size})
for p in sorted(FIG.glob('*.png')):
    index.append({'Type':'FigurePNG','File':p.name,'Bytes':p.stat().st_size})
for p in sorted(FIG.glob('*.pdf')):
    index.append({'Type':'FigurePDF','File':p.name,'Bytes':p.stat().st_size})
index_df=pd.DataFrame(index)
save_table(index_df,'Table_99_Output_Index')

bestrow=perf.sort_values('MacroF1',ascending=False).iloc[0]
summary_text = (
    f"RUN ID: {cfg_hash()}\n"
    f"DATA: {provenance}\n"
    f"PRIMARY TEST BEST METHOD BY MACRO-F1: {bestrow.Method}\n"
    f"Macro-F1: {bestrow.MacroF1:.4f}\n"
    f"Macro-AUROC: {bestrow.MacroAUROC:.4f}\n"
    f"Trust threshold: {trust_threshold:.4f}\n\n"
    f"OUTPUTS:\n"
    f"Tables: {len(list(TAB.glob('*.csv')))}\n"
    f"PNG figures: {len(list(FIG.glob('*.png')))}\n"
    f"PDF figures: {len(list(FIG.glob('*.pdf')))}\n\n"
    "IMPORTANT:\n"
    "1. Consult Table_20_Claim_Eligibility_and_Integrity_Gate.csv before manuscript writing.\n"
    "2. Synthetic smoke output is never manuscript evidence.\n"
    "3. Only independent-seed statistics in Table_18 support inferential claims.\n"
    "4. External UNSW-NB15 results are separate and never pooled with CIC-IoT-2023.\n"
)
(OUT/'MANUSCRIPT_RESULT_SUMMARY.txt').write_text(summary_text)
zip_path=shutil.make_archive(str(OUT),'zip',OUT)
print(summary_text)
print('ZIP:',zip_path)
print('Runtime seconds:',round(time.time()-RUN_START,1))


In [ ]:
# Final — download the output ZIP to your computer
import os, shutil
from pathlib import Path
from google.colab import files
out=Path(CONFIG['OUTDIR']); zip_path=Path(str(out)+'.zip')
if not zip_path.exists():
    zip_path=Path(shutil.make_archive(str(out),'zip',out))
print(f'ZIP: {zip_path} ({zip_path.stat().st_size/1024**2:.1f} MB)')
try:
    files.download(str(zip_path))
except Exception as e:
    print('Automatic download failed:',e,'- get it from Google Drive (My Drive) instead.')